In [ ]:
%run 03_build_construction_temporal_alignment.py

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

f = ROOT / "07_provenance" / "construction_temporal_alignment.csv"

qc = pd.read_csv(f)

print("Rows:", len(qc))
print("Duplicate transitions:", qc["transition"].duplicated().sum())
print("Missing values:", qc.isna().sum().sum())
print("Temporal leakage:", qc["temporal_leakage"].sum())

display(qc)

assert len(qc) == 10
assert qc["transition"].nunique() == 10
assert qc.isna().sum().sum() == 0
assert qc["temporal_leakage"].sum() == 0

expected_months = [
    "2025M08", "2025M09", "2025M10", "2025M11", "2025M12",
    "2026M01", "2026M02", "2026M03", "2026M04", "2026M05"
]

assert qc["latest_available_consent_month"].tolist() == expected_months

print("\nFINAL TEMPORAL ALIGNMENT QC: PASS")

In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

RAW_FILE = (
    ROOT
    / "01_raw"
    / "construction"
    / "statsnz_building_consents_ta_monthly_2020M01_2026M07.csv"
)

print("File exists:", RAW_FILE.exists())
print("File:", RAW_FILE)

print("\nFIRST 15 RAW LINES")
print("=" * 100)

with open(RAW_FILE, "r", encoding="utf-8-sig") as f:
    for i in range(15):
        line = f.readline()
        print(f"{i+1:02d}: {line.rstrip()}")

In [ ]:
import csv

print("RAW CSV STRUCTURE QC")
print("=" * 80)

with open(RAW_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)

    rows = []
    for i, row in enumerate(reader):
        rows.append(row)
        if i >= 7:
            break

for i, row in enumerate(rows, start=1):
    non_empty = [(j, x) for j, x in enumerate(row) if x.strip() != ""]

    print(f"\nROW {i}")
    print("Total cells:", len(row))
    print("Non-empty cells:", len(non_empty))
    print("First 20 non-empty values:")

    for col_index, value in non_empty[:20]:
        print(f"  column {col_index}: {value}")

In [ ]:
import csv
import pandas as pd

# ============================================================
# FULL RAW BUILDING CONSENTS QC
# Read-only: does NOT modify the Stats NZ raw file
# ============================================================

with open(RAW_FILE, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.reader(f))

# Header rows
ta_row = rows[1]
building_type_row = rows[2]
nature_row = rows[3]
observation_row = rows[4]

# Keep only genuine monthly observation rows.
# Stats NZ raw CSV also contains metadata/notes after the data table.
data_rows = [
    row for row in rows[5:]
    if len(row) > 0
    and row[0].strip()
    and pd.Series([row[0].strip()]).str.match(
        r"^\d{4}M(0[1-9]|1[0-2])$"
    ).iloc[0]
]

# ------------------------------------------------------------
# 1. BASIC DIMENSIONS
# ------------------------------------------------------------

n_columns = len(rows[0])
n_months = len(data_rows)

ta_names = [
    ta_row[i]
    for i in range(1, len(ta_row), 6)
]

building_types = building_type_row[1:7]

print("BUILDING CONSENTS RAW QC")
print("=" * 80)

print("Total columns:", n_columns)
print("Months:", n_months)
print("Geography categories:", len(ta_names))
print("Building types:", len(building_types))

print("\nTime range:")
print("First month:", data_rows[0][0])
print("Last month:", data_rows[-1][0])

print("\nBuilding types:")
for x in building_types:
    print(" -", x)

# ------------------------------------------------------------
# 2. STRUCTURAL QC
# ------------------------------------------------------------

bad_width_rows = [
    i + 6
    for i, row in enumerate(data_rows)
    if len(row) != 415
]

duplicate_ta = (
    len(ta_names) - len(set(ta_names))
)

months = [row[0] for row in data_rows]

duplicate_months = (
    len(months) - len(set(months))
)

# ------------------------------------------------------------
# 3. HEADER CONSISTENCY
# ------------------------------------------------------------

all_nature_new = all(
    x == "New"
    for x in nature_row[1:]
)

all_observation_number = all(
    x == "Number"
    for x in observation_row[1:]
)

expected_types = building_types

building_type_pattern_ok = True

for start in range(1, 415, 6):
    if building_type_row[start:start+6] != expected_types:
        building_type_pattern_ok = False
        break

# ------------------------------------------------------------
# 4. VALUE QC
# ------------------------------------------------------------

values = []

non_numeric = []

for r_idx, row in enumerate(data_rows, start=6):
    for c_idx, value in enumerate(row[1:], start=1):

        value = value.strip()

        try:
            values.append(float(value))

        except ValueError:
            non_numeric.append(
                (r_idx, c_idx, value)
            )

negative_values = [
    x for x in values
    if x < 0
]

zero_count = sum(
    x == 0 for x in values
)

# ------------------------------------------------------------
# 5. EXPECTED CELL COUNT
# ------------------------------------------------------------

expected_cells = (
    n_months
    * len(ta_names)
    * len(building_types)
)

actual_numeric_cells = len(values)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\nSTRUCTURE")
print("Bad-width data rows:", len(bad_width_rows))
print("Duplicate TA names:", duplicate_ta)
print("Duplicate months:", duplicate_months)
print("All Building Nature = New:", all_nature_new)
print(
    "All Observations = Number:",
    all_observation_number
)
print(
    "Building Type pattern consistent:",
    building_type_pattern_ok
)

print("\nVALUES")
print("Expected data cells:", expected_cells)
print("Numeric data cells:", actual_numeric_cells)
print("Non-numeric cells:", len(non_numeric))
print("Negative values:", len(negative_values))
print("Zero values:", zero_count)

# ------------------------------------------------------------
# HARD QC
# ------------------------------------------------------------

assert n_columns == 415
assert n_months == 79
assert len(ta_names) == 69
assert len(building_types) == 6

assert months[0] == "2020M01"
assert months[-1] == "2026M07"

assert len(bad_width_rows) == 0
assert duplicate_ta == 0
assert duplicate_months == 0

assert all_nature_new
assert all_observation_number
assert building_type_pattern_ok

assert expected_cells == 32706
assert actual_numeric_cells == 32706
assert len(non_numeric) == 0
assert len(negative_values) == 0

print("\nFULL RAW CONSTRUCTION QC: PASS")

In [ ]:
# ============================================================
# RESHAPE STATS NZ BUILDING CONSENTS
# Wide -> Long
# No modification, interpolation, imputation, or scaling
# ============================================================

records = []

for row in data_rows:

    month = row[0].strip()

    for ta_idx in range(69):

        start_col = 1 + ta_idx * 6

        ta_name = ta_row[start_col].strip()

        for type_idx in range(6):

            col = start_col + type_idx

            building_type = building_type_row[col].strip()
            value = int(row[col])

            records.append({
                "consent_month": month,
                "territorial_authority": ta_name,
                "building_type": building_type,
                "building_nature": nature_row[col].strip(),
                "observation": observation_row[col].strip(),
                "value": value
            })

construction_long = pd.DataFrame(records)

# ============================================================
# QC
# ============================================================

print("CONSTRUCTION LONG-FORMAT QC")
print("=" * 80)

print("Rows:", len(construction_long))
print("Months:", construction_long["consent_month"].nunique())
print(
    "Geography categories:",
    construction_long["territorial_authority"].nunique()
)
print(
    "Building types:",
    construction_long["building_type"].nunique()
)

print(
    "Duplicate month-TA-type:",
    construction_long.duplicated(
        subset=[
            "consent_month",
            "territorial_authority",
            "building_type"
        ]
    ).sum()
)

print(
    "Missing values:",
    construction_long.isna().sum().sum()
)

print(
    "Negative values:",
    (construction_long["value"] < 0).sum()
)

print(
    "Zero values:",
    (construction_long["value"] == 0).sum()
)

# HARD QC

assert len(construction_long) == 32706
assert construction_long["consent_month"].nunique() == 79
assert construction_long["territorial_authority"].nunique() == 69
assert construction_long["building_type"].nunique() == 6

assert construction_long.duplicated(
    subset=[
        "consent_month",
        "territorial_authority",
        "building_type"
    ]
).sum() == 0

assert construction_long.isna().sum().sum() == 0
assert (construction_long["value"] < 0).sum() == 0

print("\nLONG-FORMAT RESHAPE QC: PASS")

display(construction_long.head(12))

In [ ]:
# ============================================================
# VALUE-PRESERVATION AUDIT
# Prove that reshaping did not change any Stats NZ value
# ============================================================

# Original values directly from raw CSV, in original order
raw_values = [
    int(value.strip())
    for row in data_rows
    for value in row[1:]
]

# Values after long-format reshape, in the same order
long_values = construction_long["value"].tolist()

print("VALUE-PRESERVATION AUDIT")
print("=" * 80)

print("Raw values:", len(raw_values))
print("Long-format values:", len(long_values))

# Cell-by-cell comparison
mismatch_count = sum(
    raw != long
    for raw, long in zip(raw_values, long_values)
)

print("Value mismatches:", mismatch_count)

# Additional integrity checks
print("Raw sum:", sum(raw_values))
print("Long-format sum:", sum(long_values))

print("Raw zero count:", sum(v == 0 for v in raw_values))
print("Long zero count:", sum(v == 0 for v in long_values))

print("Raw minimum:", min(raw_values))
print("Long minimum:", min(long_values))

print("Raw maximum:", max(raw_values))
print("Long maximum:", max(long_values))

# Hard assertions
assert len(raw_values) == 32706
assert len(long_values) == 32706

assert raw_values == long_values

assert sum(raw_values) == construction_long["value"].sum()

assert sum(v == 0 for v in raw_values) == (
    construction_long["value"] == 0
).sum()

print("\nVALUE PRESERVATION: 100%")
print("RESHAPE DID NOT ALTER ANY STATS NZ VALUE")
print("VALUE-PRESERVATION AUDIT: PASS")

In [ ]:
# ============================================================
# SAVE VERIFIED LONG-FORMAT CONSTRUCTION DATA
# ============================================================

OUTPUT_DIR = ROOT / "02_intermediate" / "construction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "statsnz_building_consents_ta_monthly_long.parquet"
)

construction_long.to_parquet(
    OUTPUT_FILE,
    index=False
)

# Re-open saved file for integrity check
saved = pd.read_parquet(OUTPUT_FILE)

print("SAVED CONSTRUCTION DATA QC")
print("=" * 80)

print("File:", OUTPUT_FILE)
print("Rows:", len(saved))
print("Columns:", len(saved.columns))
print("Months:", saved["consent_month"].nunique())
print(
    "Geography categories:",
    saved["territorial_authority"].nunique()
)
print(
    "Building types:",
    saved["building_type"].nunique()
)
print(
    "Missing values:",
    saved.isna().sum().sum()
)

# Exact equality with in-memory verified dataset
exact_match = saved.equals(
    construction_long.reset_index(drop=True)
)

print("Exact dataframe match:", exact_match)

assert len(saved) == 32706
assert saved["consent_month"].nunique() == 79
assert saved["territorial_authority"].nunique() == 69
assert saved["building_type"].nunique() == 6
assert saved.isna().sum().sum() == 0
assert exact_match

print("\nINTERMEDIATE CONSTRUCTION SAVE QC: PASS")

In [ ]:
# ============================================================
# STEP 18.1 — INSPECT STATS NZ CONSTRUCTION GEOGRAPHIES
# No merge and no modification yet
# ============================================================

construction_geographies = sorted(
    construction_long["territorial_authority"]
    .unique()
    .tolist()
)

print("STATS NZ CONSTRUCTION GEOGRAPHY QC")
print("=" * 80)

print("Total geography categories:", len(construction_geographies))

print(
    "Contains New Zealand:",
    "New Zealand" in construction_geographies
)

print(
    "Contains Area Outside Territorial Authority:",
    "Area Outside Territorial Authority"
    in construction_geographies
)

actual_tas = [
    x for x in construction_geographies
    if x not in [
        "New Zealand",
        "Area Outside Territorial Authority"
    ]
]

print("Actual TA categories:", len(actual_tas))

print("\nALL 69 GEOGRAPHY CATEGORIES")
print("=" * 80)

for i, name in enumerate(construction_geographies, start=1):
    print(f"{i:02d}. {name}")

# HARD QC
assert len(construction_geographies) == 69
assert len(actual_tas) == 67
assert "New Zealand" in construction_geographies
assert "Area Outside Territorial Authority" in construction_geographies

print("\nSTATS NZ GEOGRAPHY STRUCTURE QC: PASS")

In [ ]:
print("TA NAME MISMATCH DIAGNOSTIC")
print("=" * 80)

print("Construction actual TAs:", len(construction_actual_names))
print("Airbnb actual TAs:", len(airbnb_actual_names))
print("Exact matches:", len(exact_matches))

print("\nONLY IN CONSTRUCTION")
print("-" * 80)

for x in only_in_construction:
    print(repr(x))

print("\nONLY IN AIRBNB TA2025")
print("-" * 80)

for x in only_in_airbnb:
    print(repr(x))

print("\nCOUNTS")
print("Only Construction:", len(only_in_construction))
print("Only Airbnb:", len(only_in_airbnb))

In [ ]:
# ============================================================
# STEP 18.3 — MATCH CONSTRUCTION NAMES TO OFFICIAL TA2025
# ASCII NAMES
# Read-only. No manual renaming.
# ============================================================

airbnb_ta_reference = pd.concat(
    [
        pd.read_parquet(
            f,
            columns=[
                "ta_code",
                "ta_name",
                "ta_name_ascii"
            ]
        )
        for f in geo_files
    ],
    ignore_index=True
)

# One official record per TA
ta_reference = (
    airbnb_ta_reference[
        airbnb_ta_reference["ta_code"]
        .astype(str)
        .str.zfill(3) != "999"
    ][
        ["ta_code", "ta_name", "ta_name_ascii"]
    ]
    .drop_duplicates()
    .sort_values("ta_code")
    .reset_index(drop=True)
)

construction_names = set(actual_tas)

official_ascii_names = set(
    ta_reference["ta_name_ascii"]
)

only_construction_ascii = sorted(
    construction_names - official_ascii_names
)

only_ta2025_ascii = sorted(
    official_ascii_names - construction_names
)

exact_ascii_matches = (
    construction_names & official_ascii_names
)

print("CONSTRUCTION ↔ TA2025 ASCII NAME AUDIT")
print("=" * 80)

print("TA2025 actual TAs:", len(ta_reference))
print(
    "Construction actual TAs:",
    len(construction_names)
)
print(
    "Exact ASCII matches:",
    len(exact_ascii_matches)
)

print(
    "Only in Construction:",
    len(only_construction_ascii)
)

print(
    "Only in TA2025 ASCII:",
    len(only_ta2025_ascii)
)

if only_construction_ascii:
    print("\nOnly Construction:")
    for x in only_construction_ascii:
        print(repr(x))

if only_ta2025_ascii:
    print("\nOnly TA2025 ASCII:")
    for x in only_ta2025_ascii:
        print(repr(x))

# Show the two previously identified cases
print("\nCHECK PREVIOUS NAME DIFFERENCES")
print("-" * 80)

display(
    ta_reference[
        ta_reference["ta_name_ascii"].isin(
            ["Opotiki District", "Otorohanga District"]
        )
    ]
)

# HARD QC
assert len(ta_reference) == 67
assert len(construction_names) == 67
assert len(exact_ascii_matches) == 67
assert len(only_construction_ascii) == 0
assert len(only_ta2025_ascii) == 0

print("\n67/67 CONSTRUCTION ↔ TA2025 ASCII MATCH: PASS")

In [ ]:
# ============================================================
# STEP 18.4 — BUILD OFFICIAL CONSTRUCTION ↔ TA2025 CROSSWALK
# No fuzzy matching. No manual renaming.
# ============================================================

# 67 actual TAs:
# Stats NZ construction names match official TA2025 ASCII names.
crosswalk_actual = (
    pd.DataFrame({
        "construction_ta_name": sorted(actual_tas)
    })
    .merge(
        ta_reference,
        left_on="construction_ta_name",
        right_on="ta_name_ascii",
        how="left",
        validate="one_to_one"
    )
)

# Add official outside-TA category separately
crosswalk_outside = pd.DataFrame({
    "construction_ta_name": [
        "Area Outside Territorial Authority"
    ],
    "ta_code": ["999"],
    "ta_name": [
        "Area Outside Territorial Authority"
    ],
    "ta_name_ascii": [
        "Area Outside Territorial Authority"
    ]
})

construction_ta_crosswalk = pd.concat(
    [
        crosswalk_actual,
        crosswalk_outside
    ],
    ignore_index=True
)

# Standardise code representation only in the processed crosswalk
construction_ta_crosswalk["ta_code"] = (
    construction_ta_crosswalk["ta_code"]
    .astype(str)
    .str.zfill(3)
)

# ============================================================
# QC
# ============================================================

print("CONSTRUCTION TA CROSSWALK QC")
print("=" * 80)

print("Crosswalk rows:", len(construction_ta_crosswalk))
print(
    "Unique construction names:",
    construction_ta_crosswalk[
        "construction_ta_name"
    ].nunique()
)
print(
    "Unique TA codes:",
    construction_ta_crosswalk["ta_code"].nunique()
)
print(
    "Missing values:",
    construction_ta_crosswalk.isna().sum().sum()
)
print(
    "Duplicate construction names:",
    construction_ta_crosswalk[
        "construction_ta_name"
    ].duplicated().sum()
)
print(
    "Duplicate TA codes:",
    construction_ta_crosswalk[
        "ta_code"
    ].duplicated().sum()
)

# Check special cases
display(
    construction_ta_crosswalk[
        construction_ta_crosswalk["ta_code"].isin(
            ["018", "027", "999"]
        )
    ]
)

# HARD QC
assert len(construction_ta_crosswalk) == 68
assert construction_ta_crosswalk["construction_ta_name"].nunique() == 68
assert construction_ta_crosswalk["ta_code"].nunique() == 68
assert construction_ta_crosswalk.isna().sum().sum() == 0

assert construction_ta_crosswalk[
    "construction_ta_name"
].duplicated().sum() == 0

assert construction_ta_crosswalk[
    "ta_code"
].duplicated().sum() == 0

assert (
    "New Zealand"
    not in construction_ta_crosswalk["construction_ta_name"].values
)

assert (
    construction_ta_crosswalk.loc[
        construction_ta_crosswalk["construction_ta_name"]
        == "Area Outside Territorial Authority",
        "ta_code"
    ].iloc[0]
    == "999"
)

print("\nCONSTRUCTION TA CROSSWALK QC: PASS")

In [ ]:
# ============================================================
# STEP 18.5 — AUDIT STATS NZ AREA OUTSIDE TA (999)
# Read-only. Do not modify any values.
# ============================================================

outside = construction_long[
    construction_long["territorial_authority"]
    == "Area Outside Territorial Authority"
].copy()

print("AREA OUTSIDE TERRITORIAL AUTHORITY (TA999) AUDIT")
print("=" * 80)

print("Rows:", len(outside))
print("Months:", outside["consent_month"].nunique())
print("Building types:", outside["building_type"].nunique())
print("Missing values:", outside["value"].isna().sum())
print("Zero values:", (outside["value"] == 0).sum())
print("Non-zero values:", (outside["value"] != 0).sum())
print("Total value:", outside["value"].sum())

# Summary by building type
outside_summary = (
    outside
    .groupby("building_type")["value"]
    .agg(
        months="size",
        nonzero_months=lambda x: (x != 0).sum(),
        total="sum",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

print("\nSUMMARY BY BUILDING TYPE")
display(outside_summary)

# Show only months where any outside-TA value is non-zero
outside_nonzero = outside[
    outside["value"] != 0
].sort_values(
    ["consent_month", "building_type"]
)

print("\nNON-ZERO OUTSIDE-TA RECORDS")
print("Rows:", len(outside_nonzero))

display(outside_nonzero.head(30))

# Structural QC
assert len(outside) == 79 * 6
assert outside["consent_month"].nunique() == 79
assert outside["building_type"].nunique() == 6
assert outside["value"].isna().sum() == 0

print("\nTA999 STRUCTURAL QC: PASS")

In [ ]:
# ============================================================
# STEP 18.6 — FINALISE CONSTRUCTION TA CROSSWALK
# Preserve TA999 but exclude it from listing-level
# construction-context assignment
# ============================================================

construction_ta_crosswalk_final = (
    construction_ta_crosswalk.copy()
)

construction_ta_crosswalk_final[
    "construction_context_eligible"
] = (
    construction_ta_crosswalk_final["ta_code"] != "999"
)

construction_ta_crosswalk_final[
    "construction_context_note"
] = "Actual Territorial Authority"

construction_ta_crosswalk_final.loc[
    construction_ta_crosswalk_final["ta_code"] == "999",
    "construction_context_note"
] = (
    "Outside-TA aggregate; retained from source but "
    "not used for listing-level construction context"
)

# QC
print("FINAL CONSTRUCTION TA CROSSWALK")
print("=" * 80)

print(
    "Rows:",
    len(construction_ta_crosswalk_final)
)

print(
    "Eligible actual TAs:",
    construction_ta_crosswalk_final[
        "construction_context_eligible"
    ].sum()
)

print(
    "Excluded categories:",
    (
        ~construction_ta_crosswalk_final[
            "construction_context_eligible"
        ]
    ).sum()
)

display(
    construction_ta_crosswalk_final[
        construction_ta_crosswalk_final["ta_code"].isin(
            ["018", "027", "999"]
        )
    ]
)

assert len(construction_ta_crosswalk_final) == 68

assert (
    construction_ta_crosswalk_final[
        "construction_context_eligible"
    ].sum()
    == 67
)

assert (
    construction_ta_crosswalk_final.loc[
        construction_ta_crosswalk_final["ta_code"] == "999",
        "construction_context_eligible"
    ].iloc[0]
    == False
)

print("\nFINAL CONSTRUCTION TA CROSSWALK QC: PASS")

In [ ]:
# ============================================================
# STEP 18.7 — SAVE CROSSWALK + ADD OFFICIAL TA CODE
# No modification of Stats NZ consent values
# ============================================================

OUTPUT_DIR = ROOT / "02_intermediate" / "construction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Save audited TA crosswalk
# ------------------------------------------------------------

CROSSWALK_FILE = (
    OUTPUT_DIR
    / "construction_ta2025_crosswalk.parquet"
)

construction_ta_crosswalk_final.to_parquet(
    CROSSWALK_FILE,
    index=False
)

# ------------------------------------------------------------
# 2. Remove national aggregate from listing-level dataset
#    Keep raw/intermediate source file unchanged.
# ------------------------------------------------------------

construction_ta = construction_long[
    construction_long["territorial_authority"] != "New Zealand"
].copy()

# ------------------------------------------------------------
# 3. Attach official TA2025 code through audited crosswalk
# ------------------------------------------------------------

construction_ta = construction_ta.merge(
    construction_ta_crosswalk_final[
        [
            "construction_ta_name",
            "ta_code",
            "ta_name",
            "ta_name_ascii",
            "construction_context_eligible"
        ]
    ],
    left_on="territorial_authority",
    right_on="construction_ta_name",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 4. QC
# ------------------------------------------------------------

print("CONSTRUCTION + TA2025 CODE QC")
print("=" * 80)

print("Rows:", len(construction_ta))
print("Months:", construction_ta["consent_month"].nunique())
print("TA codes:", construction_ta["ta_code"].nunique())
print("Building types:", construction_ta["building_type"].nunique())

print(
    "Missing TA codes:",
    construction_ta["ta_code"].isna().sum()
)

print(
    "Eligible rows:",
    construction_ta["construction_context_eligible"].sum()
)

print(
    "TA999 rows:",
    (construction_ta["ta_code"] == "999").sum()
)

print(
    "Consent-value sum before:",
    construction_long.loc[
        construction_long["territorial_authority"] != "New Zealand",
        "value"
    ].sum()
)

print(
    "Consent-value sum after:",
    construction_ta["value"].sum()
)

# Expected:
# 79 months × 68 geographic categories × 6 building types
expected_rows = 79 * 68 * 6

assert len(construction_ta) == expected_rows
assert construction_ta["consent_month"].nunique() == 79
assert construction_ta["ta_code"].nunique() == 68
assert construction_ta["building_type"].nunique() == 6

assert construction_ta["ta_code"].isna().sum() == 0
assert (construction_ta["ta_code"] == "999").sum() == 79 * 6

assert (
    construction_ta["value"].sum()
    ==
    construction_long.loc[
        construction_long["territorial_authority"] != "New Zealand",
        "value"
    ].sum()
)

print("\nCONSTRUCTION + TA2025 CODE QC: PASS")

# ------------------------------------------------------------
# 5. Save and reopen
# ------------------------------------------------------------

TA_FILE = (
    OUTPUT_DIR
    / "statsnz_building_consents_ta2025_monthly.parquet"
)

construction_ta.to_parquet(
    TA_FILE,
    index=False
)

saved_ta = pd.read_parquet(TA_FILE)

assert saved_ta.equals(
    construction_ta.reset_index(drop=True)
)

print("SAVED FILE EXACT MATCH: PASS")
print("\nSaved:", TA_FILE)
print("Crosswalk:", CROSSWALK_FILE)

In [ ]:
# ============================================================
# STEP 19.1 — TEMPORAL COVERAGE AUDIT
# Check the 10 frozen consent months across
# 67 eligible TAs × 6 building types
# No feature engineering yet.
# ============================================================

# The 10 consent months already frozen by temporal alignment
frozen_months = [
    "2025M08",
    "2025M09",
    "2025M10",
    "2025M11",
    "2025M12",
    "2026M01",
    "2026M02",
    "2026M03",
    "2026M04",
    "2026M05"
]

# Keep ONLY the 67 actual TAs eligible for local
# construction context.
eligible = construction_ta[
    construction_ta["construction_context_eligible"] == True
].copy()

study_construction = eligible[
    eligible["consent_month"].isin(frozen_months)
].copy()

print("FROZEN CONSTRUCTION TEMPORAL COVERAGE AUDIT")
print("=" * 80)

print("Frozen consent months:", len(frozen_months))
print("Rows found:", len(study_construction))
print("Unique months:", study_construction["consent_month"].nunique())
print("Unique TA codes:", study_construction["ta_code"].nunique())
print("Building types:", study_construction["building_type"].nunique())

# ------------------------------------------------------------
# Expected structure
# 10 months × 67 TAs × 6 building types
# ------------------------------------------------------------

expected_rows = 10 * 67 * 6

print("\nExpected rows:", expected_rows)
print("Actual rows:", len(study_construction))

# ------------------------------------------------------------
# Check each month independently
# ------------------------------------------------------------

coverage = (
    study_construction
    .groupby("consent_month")
    .agg(
        rows=("value", "size"),
        ta_codes=("ta_code", "nunique"),
        building_types=("building_type", "nunique"),
        missing_values=("value", lambda x: x.isna().sum())
    )
    .reindex(frozen_months)
    .reset_index()
)

coverage["expected_rows"] = 67 * 6

coverage["complete"] = (
    (coverage["rows"] == 67 * 6)
    & (coverage["ta_codes"] == 67)
    & (coverage["building_types"] == 6)
    & (coverage["missing_values"] == 0)
)

display(coverage)

# ------------------------------------------------------------
# Duplicate key check
# ------------------------------------------------------------

duplicate_keys = study_construction.duplicated(
    subset=[
        "consent_month",
        "ta_code",
        "building_type"
    ]
).sum()

print("\nDuplicate month-TA-type keys:", duplicate_keys)

# ------------------------------------------------------------
# Check missing combinations explicitly
# ------------------------------------------------------------

expected_index = pd.MultiIndex.from_product(
    [
        frozen_months,
        sorted(eligible["ta_code"].unique()),
        sorted(eligible["building_type"].unique())
    ],
    names=[
        "consent_month",
        "ta_code",
        "building_type"
    ]
)

actual_index = pd.MultiIndex.from_frame(
    study_construction[
        [
            "consent_month",
            "ta_code",
            "building_type"
        ]
    ]
)

missing_combinations = expected_index.difference(actual_index)

print("Missing month-TA-type combinations:",
      len(missing_combinations))

if len(missing_combinations) > 0:
    print("\nFIRST MISSING COMBINATIONS")
    display(
        pd.DataFrame(
            list(missing_combinations),
            columns=[
                "consent_month",
                "ta_code",
                "building_type"
            ]
        ).head(30)
    )

# ------------------------------------------------------------
# HARD QC
# ------------------------------------------------------------

assert len(study_construction) == 10 * 67 * 6
assert study_construction["consent_month"].nunique() == 10
assert study_construction["ta_code"].nunique() == 67
assert study_construction["building_type"].nunique() == 6

assert duplicate_keys == 0
assert len(missing_combinations) == 0
assert study_construction["value"].isna().sum() == 0
assert coverage["complete"].all()

print("\n10/10 MONTHS COMPLETE: PASS")
print("67/67 ELIGIBLE TAs COMPLETE: PASS")
print("6/6 BUILDING TYPES COMPLETE: PASS")
print("MISSING COMBINATIONS: 0")
print("\nFROZEN CONSTRUCTION TEMPORAL COVERAGE QC: PASS")

In [ ]:
# ============================================================
# STEP 19.2 — HISTORICAL WINDOW COVERAGE AUDIT
# Verify complete 3-, 6-, and 12-month histories
# BEFORE feature engineering.
# ============================================================

import pandas as pd

# Work only with 67 eligible actual TAs
eligible_hist = construction_ta[
    construction_ta["construction_context_eligible"] == True
].copy()

# Convert YYYYMMM safely to monthly Period
eligible_hist["consent_period"] = pd.PeriodIndex(
    pd.to_datetime(
        eligible_hist["consent_month"],
        format="%YM%m"
    ),
    freq="M"
)

# Frozen months used by T1-T10
frozen_periods = pd.PeriodIndex(
    pd.to_datetime(
        frozen_months,
        format="%YM%m"
    ),
    freq="M"
)

window_results = []

for anchor_period in frozen_periods:

    for window in [3, 6, 12]:

        # Inclusive trailing window:
        # e.g. 3-month at 2025M08 =
        # 2025M06, 2025M07, 2025M08
        start_period = anchor_period - (window - 1)

        subset = eligible_hist[
            (eligible_hist["consent_period"] >= start_period)
            &
            (eligible_hist["consent_period"] <= anchor_period)
        ]

        expected_rows = window * 67 * 6

        actual_months = subset["consent_period"].nunique()
        actual_tas = subset["ta_code"].nunique()
        actual_types = subset["building_type"].nunique()

        duplicate_keys = subset.duplicated(
            subset=[
                "consent_period",
                "ta_code",
                "building_type"
            ]
        ).sum()

        missing_values = subset["value"].isna().sum()

        complete = (
            len(subset) == expected_rows
            and actual_months == window
            and actual_tas == 67
            and actual_types == 6
            and duplicate_keys == 0
            and missing_values == 0
        )

        window_results.append({
            "anchor_consent_month":
                anchor_period.strftime("%YM%m"),
            "window_months": window,
            "window_start":
                start_period.strftime("%YM%m"),
            "window_end":
                anchor_period.strftime("%YM%m"),
            "expected_rows": expected_rows,
            "actual_rows": len(subset),
            "actual_months": actual_months,
            "ta_codes": actual_tas,
            "building_types": actual_types,
            "duplicate_keys": duplicate_keys,
            "missing_values": missing_values,
            "complete": complete
        })

window_qc = pd.DataFrame(window_results)

print("HISTORICAL CONSTRUCTION WINDOW COVERAGE")
print("=" * 80)

display(window_qc)

print("\nSUMMARY")
print("-" * 80)

print("Windows checked:", len(window_qc))
print("Complete windows:", window_qc["complete"].sum())
print("Incomplete windows:", (~window_qc["complete"]).sum())

# Hard QC
assert len(window_qc) == 30
assert window_qc["complete"].all()
assert window_qc["missing_values"].sum() == 0
assert window_qc["duplicate_keys"].sum() == 0

print("\n3-MONTH WINDOWS: PASS")
print("6-MONTH WINDOWS: PASS")
print("12-MONTH WINDOWS: PASS")
print("\nHISTORICAL WINDOW COVERAGE QC: PASS")

In [ ]:
# ============================================================
# STEP 20.1 — RESIDENTIAL COMPOSITION ACCOUNTING AUDIT
#
# Goal:
# Check whether Stats NZ "Dwelling units" equals the sum of
# its four residential dwelling-type components.
#
# Read-only QC:
# - no imputation
# - no interpolation
# - no modification of source values
# - no feature engineering yet
# ============================================================

import pandas as pd

# Use ONLY the 67 actual Territorial Authorities.
# TA999 is excluded from local construction context.
residential_qc = construction_ta[
    construction_ta["construction_context_eligible"] == True
].copy()

# ------------------------------------------------------------
# 1. Keep the five residential series
# ------------------------------------------------------------

residential_types = [
    "Dwelling units",
    "Houses",
    "Apartments",
    "Retirement village units",
    "Townhouses, flats, units, and other dwellings"
]

residential_qc = residential_qc[
    residential_qc["building_type"].isin(residential_types)
].copy()

# ------------------------------------------------------------
# 2. Pivot to one row per month × TA
# ------------------------------------------------------------

residential_wide = (
    residential_qc
    .pivot(
        index=[
            "consent_month",
            "ta_code"
        ],
        columns="building_type",
        values="value"
    )
    .reset_index()
)

residential_wide.columns.name = None

# ------------------------------------------------------------
# 3. Calculate component sum
# ------------------------------------------------------------

component_columns = [
    "Houses",
    "Apartments",
    "Retirement village units",
    "Townhouses, flats, units, and other dwellings"
]

residential_wide["component_sum"] = (
    residential_wide[component_columns].sum(axis=1)
)

residential_wide["difference"] = (
    residential_wide["Dwelling units"]
    - residential_wide["component_sum"]
)

residential_wide["exact_match"] = (
    residential_wide["difference"] == 0
)

# ------------------------------------------------------------
# 4. QC results
# ------------------------------------------------------------

print("RESIDENTIAL COMPOSITION ACCOUNTING AUDIT")
print("=" * 80)

print("Rows checked:", len(residential_wide))
print(
    "Months:",
    residential_wide["consent_month"].nunique()
)
print(
    "TA codes:",
    residential_wide["ta_code"].nunique()
)

print(
    "Missing values:",
    residential_wide[
        residential_types
    ].isna().sum().sum()
)

print("\nACCOUNTING RELATIONSHIP")
print("-" * 80)

print(
    "Exact matches:",
    residential_wide["exact_match"].sum()
)

print(
    "Mismatches:",
    (~residential_wide["exact_match"]).sum()
)

print(
    "Match rate (%):",
    round(
        residential_wide["exact_match"].mean() * 100,
        4
    )
)

print(
    "Total Dwelling units:",
    residential_wide["Dwelling units"].sum()
)

print(
    "Total component sum:",
    residential_wide["component_sum"].sum()
)

print(
    "Total difference:",
    residential_wide["difference"].sum()
)

print(
    "Minimum difference:",
    residential_wide["difference"].min()
)

print(
    "Maximum difference:",
    residential_wide["difference"].max()
)

# ------------------------------------------------------------
# 5. Inspect mismatches if any
# ------------------------------------------------------------

mismatches = residential_wide[
    residential_wide["difference"] != 0
].copy()

print("\nMISMATCH RECORDS:", len(mismatches))

if len(mismatches) > 0:

    display(
        mismatches[
            [
                "consent_month",
                "ta_code",
                "Dwelling units",
                "Houses",
                "Apartments",
                "Retirement village units",
                "Townhouses, flats, units, and other dwellings",
                "component_sum",
                "difference"
            ]
        ].head(30)
    )

# ------------------------------------------------------------
# 6. Structural QC only
#
# IMPORTANT:
# We do NOT assert that accounting equality must hold.
# We first observe what Stats NZ data actually contain.
# ------------------------------------------------------------

expected_rows = 79 * 67

assert len(residential_wide) == expected_rows
assert residential_wide["consent_month"].nunique() == 79
assert residential_wide["ta_code"].nunique() == 67

assert (
    residential_wide[
        residential_types
    ].isna().sum().sum()
    == 0
)

print("\nRESIDENTIAL COMPOSITION STRUCTURAL QC: PASS")

In [ ]:
# ============================================================
# STEP 20.2 — CONSTRUCTION DISTRIBUTION AUDIT
# 67 actual TAs only
# No transformation or feature engineering yet
# ============================================================

import pandas as pd

dist_data = construction_ta[
    construction_ta["construction_context_eligible"] == True
].copy()

# ------------------------------------------------------------
# Summary by building type
# ------------------------------------------------------------

distribution_summary = (
    dist_data
    .groupby("building_type")["value"]
    .agg(
        observations="size",
        zero_count=lambda x: (x == 0).sum(),
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

distribution_summary["zero_pct"] = (
    distribution_summary["zero_count"]
    / distribution_summary["observations"]
    * 100
)

# Add percentiles
percentiles = (
    dist_data
    .groupby("building_type")["value"]
    .quantile([0.25, 0.75, 0.90, 0.95, 0.99])
    .unstack()
    .reset_index()
    .rename(
        columns={
            0.25: "p25",
            0.75: "p75",
            0.90: "p90",
            0.95: "p95",
            0.99: "p99"
        }
    )
)

distribution_summary = distribution_summary.merge(
    percentiles,
    on="building_type",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("CONSTRUCTION DISTRIBUTION AUDIT")
print("=" * 80)

display(
    distribution_summary[
        [
            "building_type",
            "observations",
            "zero_count",
            "zero_pct",
            "mean",
            "median",
            "p25",
            "p75",
            "p90",
            "p95",
            "p99",
            "maximum"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# Specifically inspect core Dwelling units
# ------------------------------------------------------------

dwelling = dist_data[
    dist_data["building_type"] == "Dwelling units"
]["value"]

print("\nDWELLING UNITS")
print("-" * 80)

print("Observations:", len(dwelling))
print("Zero observations:", (dwelling == 0).sum())
print(
    "Zero percentage:",
    round((dwelling == 0).mean() * 100, 2)
)
print("Mean:", round(dwelling.mean(), 2))
print("Median:", dwelling.median())
print("90th percentile:", dwelling.quantile(0.90))
print("95th percentile:", dwelling.quantile(0.95))
print("99th percentile:", dwelling.quantile(0.99))
print("Maximum:", dwelling.max())

# Structural QC
assert len(dwelling) == 79 * 67
assert len(dist_data) == 79 * 67 * 6
assert dist_data["value"].isna().sum() == 0
assert (dist_data["value"] < 0).sum() == 0

print("\nCONSTRUCTION DISTRIBUTION STRUCTURAL QC: PASS")

In [ ]:
# ============================================================
# STEP 20.3 — TA-LEVEL CONSTRUCTION CONCENTRATION AUDIT
#
# Goal:
# Determine how strongly raw dwelling-consent counts are
# concentrated in a small number of Territorial Authorities.
#
# No transformation or feature engineering yet.
# ============================================================

dwelling_ta = construction_ta[
    (construction_ta["construction_context_eligible"] == True)
    &
    (construction_ta["building_type"] == "Dwelling units")
].copy()

# Total and monthly distribution for each TA
ta_summary = (
    dwelling_ta
    .groupby(["ta_code", "ta_name"])["value"]
    .agg(
        total_consents="sum",
        mean_monthly="mean",
        median_monthly="median",
        maximum_monthly="max",
        zero_months=lambda x: (x == 0).sum()
    )
    .reset_index()
)

grand_total = ta_summary["total_consents"].sum()

ta_summary["share_of_all_consents_pct"] = (
    ta_summary["total_consents"]
    / grand_total
    * 100
)

ta_summary = ta_summary.sort_values(
    "total_consents",
    ascending=False
).reset_index(drop=True)

ta_summary["cumulative_share_pct"] = (
    ta_summary["share_of_all_consents_pct"].cumsum()
)

print("TA-LEVEL DWELLING CONSENT CONCENTRATION AUDIT")
print("=" * 80)

print("TAs:", len(ta_summary))
print("Months per TA:", dwelling_ta["consent_month"].nunique())
print("Total dwelling units:", grand_total)

print("\nTOP 15 TAs")
display(
    ta_summary.head(15).round(2)
)

# Concentration statistics
top1_share = ta_summary.iloc[:1]["share_of_all_consents_pct"].sum()
top5_share = ta_summary.iloc[:5]["share_of_all_consents_pct"].sum()
top10_share = ta_summary.iloc[:10]["share_of_all_consents_pct"].sum()
top20_share = ta_summary.iloc[:20]["share_of_all_consents_pct"].sum()

print("\nCONCENTRATION")
print("-" * 80)

print("Top 1 TA share (%):", round(top1_share, 2))
print("Top 5 TA share (%):", round(top5_share, 2))
print("Top 10 TA share (%):", round(top10_share, 2))
print("Top 20 TA share (%):", round(top20_share, 2))

print("\nCROSS-TA MONTHLY SCALE")
print("-" * 80)

print(
    "Median of TA mean-monthly consents:",
    round(ta_summary["mean_monthly"].median(), 2)
)

print(
    "Maximum TA mean-monthly consents:",
    round(ta_summary["mean_monthly"].max(), 2)
)

print(
    "Minimum TA mean-monthly consents:",
    round(ta_summary["mean_monthly"].min(), 2)
)

# Structural QC
assert len(ta_summary) == 67

assert (
    dwelling_ta.groupby("ta_code").size() == 79
).all()

assert grand_total == 269339

assert round(
    ta_summary["share_of_all_consents_pct"].sum(),
    8
) == 100.0

print("\nTA-LEVEL CONCENTRATION STRUCTURAL QC: PASS")

In [ ]:
# ============================================================
# STEP 20.4 — ROLLING RESIDENTIAL SPARSITY AUDIT
#
# Goal:
# Examine 3-, 6-, and 12-month aggregated residential
# construction counts at the 10 frozen consent months.
#
# This is QC only:
# - no model features saved
# - no imputation
# - no modification of Stats NZ values
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Residential data — 67 eligible actual TAs only
# ------------------------------------------------------------

res_types = [
    "Dwelling units",
    "Houses",
    "Apartments",
    "Retirement village units",
    "Townhouses, flats, units, and other dwellings"
]

roll_source = construction_ta[
    (construction_ta["construction_context_eligible"] == True)
    &
    (construction_ta["building_type"].isin(res_types))
].copy()

roll_source["consent_period"] = pd.PeriodIndex(
    pd.to_datetime(
        roll_source["consent_month"],
        format="%YM%m"
    ),
    freq="M"
)

frozen_periods = pd.PeriodIndex(
    pd.to_datetime(
        frozen_months,
        format="%YM%m"
    ),
    freq="M"
)

# ------------------------------------------------------------
# 2. Calculate trailing totals for QC
# ------------------------------------------------------------

rolling_records = []

for anchor in frozen_periods:

    for window in [3, 6, 12]:

        start = anchor - (window - 1)

        subset = roll_source[
            (roll_source["consent_period"] >= start)
            &
            (roll_source["consent_period"] <= anchor)
        ]

        aggregated = (
            subset
            .groupby(
                ["ta_code", "building_type"],
                as_index=False
            )["value"]
            .sum()
        )

        aggregated["anchor_consent_month"] = (
            anchor.strftime("%YM%m")
        )

        aggregated["window_months"] = window

        rolling_records.append(aggregated)

rolling_qc_data = pd.concat(
    rolling_records,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Sparsity summary
# ------------------------------------------------------------

rolling_sparsity = (
    rolling_qc_data
    .groupby(
        ["window_months", "building_type"]
    )["value"]
    .agg(
        observations="size",
        zero_count=lambda x: (x == 0).sum(),
        mean="mean",
        median="median",
        maximum="max"
    )
    .reset_index()
)

rolling_sparsity["zero_pct"] = (
    rolling_sparsity["zero_count"]
    /
    rolling_sparsity["observations"]
    * 100
)

print("ROLLING RESIDENTIAL SPARSITY AUDIT")
print("=" * 80)

display(
    rolling_sparsity[
        [
            "window_months",
            "building_type",
            "observations",
            "zero_count",
            "zero_pct",
            "mean",
            "median",
            "maximum"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 4. Structural QC
#
# For each window:
# 10 anchors × 67 TAs × 5 residential types
# = 3,350 records
# ------------------------------------------------------------

print("\nSTRUCTURAL CHECK")
print("-" * 80)

for window in [3, 6, 12]:

    temp = rolling_qc_data[
        rolling_qc_data["window_months"] == window
    ]

    print(
        f"{window}-month records:",
        len(temp)
    )

    assert len(temp) == 10 * 67 * 5

# Every building type should have:
# 10 anchors × 67 TAs = 670 observations
assert (
    rolling_sparsity["observations"] == 670
).all()

assert rolling_qc_data["value"].isna().sum() == 0
assert (rolling_qc_data["value"] < 0).sum() == 0

print("\nROLLING RESIDENTIAL SPARSITY QC: PASS")

In [ ]:
# ============================================================
# STEP 21.1 — BUILD TA × TRANSITION CONSTRUCTION BACKBONE
#
# Purpose:
# Create exactly one row for every:
#     T1–T10 × 67 eligible Territorial Authorities
#
# No construction features are calculated yet.
# No Airbnb listings are merged yet.
# No Stats NZ values are modified.
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Frozen temporal mapping
#    This is the mapping already verified in temporal QC.
# ------------------------------------------------------------

transition_mapping = pd.DataFrame({
    "transition": [
        "T1", "T2", "T3", "T4", "T5",
        "T6", "T7", "T8", "T9", "T10"
    ],

    "airbnb_anchor_date": pd.to_datetime([
        "2025-10-05",
        "2025-11-07",
        "2025-12-11",
        "2026-01-16",
        "2026-02-13",
        "2026-03-17",
        "2026-04-16",
        "2026-05-23",
        "2026-06-19",
        "2026-07-12"
    ]),

    "latest_available_consent_month": [
        "2025M08",
        "2025M09",
        "2025M10",
        "2025M11",
        "2025M12",
        "2026M01",
        "2026M02",
        "2026M03",
        "2026M04",
        "2026M05"
    ]
})

# ------------------------------------------------------------
# 2. Get the 67 eligible actual TAs
# ------------------------------------------------------------

eligible_tas = (
    construction_ta[
        construction_ta["construction_context_eligible"] == True
    ][
        [
            "ta_code",
            "ta_name",
            "ta_name_ascii"
        ]
    ]
    .drop_duplicates()
    .sort_values("ta_code")
    .reset_index(drop=True)
)

print("Eligible actual TAs:", len(eligible_tas))

# ------------------------------------------------------------
# 3. Cross join:
#    10 transitions × 67 TAs = 670 rows
# ------------------------------------------------------------

construction_backbone = (
    transition_mapping
    .merge(
        eligible_tas,
        how="cross"
    )
)

# ------------------------------------------------------------
# 4. Add consent period for later rolling calculations
# ------------------------------------------------------------

construction_backbone[
    "latest_available_consent_period"
] = pd.PeriodIndex(
    pd.to_datetime(
        construction_backbone[
            "latest_available_consent_month"
        ],
        format="%YM%m"
    ),
    freq="M"
)

# ------------------------------------------------------------
# 5. Display basic structure
# ------------------------------------------------------------

print("\nCONSTRUCTION FEATURE BACKBONE")
print("=" * 80)

print("Rows:", len(construction_backbone))

print(
    "Transitions:",
    construction_backbone["transition"].nunique()
)

print(
    "TA codes:",
    construction_backbone["ta_code"].nunique()
)

print(
    "Consent months:",
    construction_backbone[
        "latest_available_consent_month"
    ].nunique()
)

print("\nFIRST 15 ROWS")

display(
    construction_backbone.head(15)
)

# ------------------------------------------------------------
# 6. Check rows per transition
# ------------------------------------------------------------

rows_per_transition = (
    construction_backbone
    .groupby(
        [
            "transition",
            "airbnb_anchor_date",
            "latest_available_consent_month"
        ],
        as_index=False
    )
    .agg(
        ta_rows=("ta_code", "size"),
        unique_tas=("ta_code", "nunique")
    )
)

print("\nROWS PER TRANSITION")
print("=" * 80)

display(rows_per_transition)

# ------------------------------------------------------------
# 7. Duplicate check
# ------------------------------------------------------------

duplicate_keys = construction_backbone.duplicated(
    subset=[
        "transition",
        "ta_code"
    ]
).sum()

print(
    "\nDuplicate transition-TA keys:",
    duplicate_keys
)

# ------------------------------------------------------------
# 8. TA999 must NOT be present
# ------------------------------------------------------------

ta999_rows = (
    construction_backbone["ta_code"] == "999"
).sum()

print(
    "TA999 rows:",
    ta999_rows
)

# ------------------------------------------------------------
# 9. Hard QC
# ------------------------------------------------------------

assert len(eligible_tas) == 67

assert len(construction_backbone) == 10 * 67

assert (
    construction_backbone["transition"].nunique()
    == 10
)

assert (
    construction_backbone["ta_code"].nunique()
    == 67
)

assert (
    construction_backbone[
        "latest_available_consent_month"
    ].nunique()
    == 10
)

assert (
    rows_per_transition["ta_rows"] == 67
).all()

assert (
    rows_per_transition["unique_tas"] == 67
).all()

assert duplicate_keys == 0

assert ta999_rows == 0

assert (
    construction_backbone.isna().sum().sum()
    == 0
)

print("\nEXPECTED ROWS: 670")
print("ACTUAL ROWS:", len(construction_backbone))

print("\n10/10 TRANSITIONS: PASS")
print("67/67 ELIGIBLE TAs: PASS")
print("DUPLICATE TRANSITION-TA KEYS: 0")
print("TA999 INCLUDED: NO")

print(
    "\nCONSTRUCTION FEATURE BACKBONE QC: PASS"
)

In [ ]:
# ============================================================
# STEP 21.2 — CALCULATE RAW 3M AND 12M RESIDENTIAL TOTALS
#
# For each of the 670 TA × transition rows:
#   - use ONLY months up to latest_available_consent_month
#   - calculate trailing 3-month totals
#   - calculate trailing 12-month totals
#
# No log transformation yet.
# No momentum yet.
# No Airbnb merge yet.
# No source values are modified.
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Prepare residential source data
# ------------------------------------------------------------

residential_types = [
    "Dwelling units",
    "Houses",
    "Apartments",
    "Retirement village units",
    "Townhouses, flats, units, and other dwellings"
]

feature_source = construction_ta[
    (construction_ta["construction_context_eligible"] == True)
    &
    (construction_ta["building_type"].isin(residential_types))
].copy()

feature_source["consent_period"] = pd.PeriodIndex(
    pd.to_datetime(
        feature_source["consent_month"],
        format="%YM%m"
    ),
    freq="M"
)

# ------------------------------------------------------------
# 2. Calculate raw rolling totals for each backbone row
# ------------------------------------------------------------

feature_records = []

for _, row in construction_backbone.iterrows():

    transition = row["transition"]
    ta_code = row["ta_code"]
    anchor_period = row["latest_available_consent_period"]

    record = {
        "transition": transition,
        "ta_code": ta_code
    }

    # --------------------------------------------------------
    # Calculate both 3-month and 12-month windows
    # --------------------------------------------------------

    for window in [3, 12]:

        start_period = anchor_period - (window - 1)

        subset = feature_source[
            (feature_source["ta_code"] == ta_code)
            &
            (feature_source["consent_period"] >= start_period)
            &
            (feature_source["consent_period"] <= anchor_period)
        ]

        # Hard check:
        # each window must contain:
        # window months × 5 residential building types
        expected_source_rows = window * 5

        assert len(subset) == expected_source_rows

        totals = (
            subset
            .groupby("building_type")["value"]
            .sum()
        )

        assert len(totals) == 5

        # Store raw totals
        record[f"dwelling_{window}m"] = totals["Dwelling units"]
        record[f"houses_{window}m"] = totals["Houses"]
        record[f"apartments_{window}m"] = totals["Apartments"]

        record[f"retirement_{window}m"] = (
            totals["Retirement village units"]
        )

        record[f"townhouse_other_{window}m"] = (
            totals[
                "Townhouses, flats, units, and other dwellings"
            ]
        )

    feature_records.append(record)

# ------------------------------------------------------------
# 3. Convert to DataFrame
# ------------------------------------------------------------

raw_construction_features = pd.DataFrame(
    feature_records
)

# ------------------------------------------------------------
# 4. Merge raw totals back onto the 670-row backbone
# ------------------------------------------------------------

construction_features_raw = (
    construction_backbone
    .merge(
        raw_construction_features,
        on=[
            "transition",
            "ta_code"
        ],
        how="left",
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 5. Basic QC
# ------------------------------------------------------------

print("RAW CONSTRUCTION ROLLING FEATURES")
print("=" * 80)

print(
    "Rows:",
    len(construction_features_raw)
)

print(
    "Transitions:",
    construction_features_raw[
        "transition"
    ].nunique()
)

print(
    "TA codes:",
    construction_features_raw[
        "ta_code"
    ].nunique()
)

raw_feature_columns = [
    "dwelling_3m",
    "houses_3m",
    "apartments_3m",
    "retirement_3m",
    "townhouse_other_3m",

    "dwelling_12m",
    "houses_12m",
    "apartments_12m",
    "retirement_12m",
    "townhouse_other_12m"
]

print(
    "Raw rolling variables:",
    len(raw_feature_columns)
)

print(
    "Missing raw values:",
    construction_features_raw[
        raw_feature_columns
    ].isna().sum().sum()
)

print(
    "Negative raw values:",
    (
        construction_features_raw[
            raw_feature_columns
        ] < 0
    ).sum().sum()
)

# ------------------------------------------------------------
# 6. Accounting check
#
# Dwelling must equal:
# houses + apartments + retirement + townhouse/other
# for BOTH windows.
# ------------------------------------------------------------

for window in [3, 12]:

    component_sum = (
        construction_features_raw[f"houses_{window}m"]
        +
        construction_features_raw[f"apartments_{window}m"]
        +
        construction_features_raw[f"retirement_{window}m"]
        +
        construction_features_raw[f"townhouse_other_{window}m"]
    )

    difference = (
        construction_features_raw[f"dwelling_{window}m"]
        -
        component_sum
    )

    print(
        f"\n{window}M accounting mismatches:",
        (difference != 0).sum()
    )

    print(
        f"{window}M maximum absolute difference:",
        difference.abs().max()
    )

    assert (difference == 0).all()

# ------------------------------------------------------------
# 7. Key uniqueness
# ------------------------------------------------------------

duplicate_keys = construction_features_raw.duplicated(
    subset=[
        "transition",
        "ta_code"
    ]
).sum()

print(
    "\nDuplicate transition-TA keys:",
    duplicate_keys
)

# ------------------------------------------------------------
# 8. Hard QC
# ------------------------------------------------------------

assert len(construction_features_raw) == 670

assert (
    construction_features_raw["transition"].nunique()
    == 10
)

assert (
    construction_features_raw["ta_code"].nunique()
    == 67
)

assert duplicate_keys == 0

assert (
    construction_features_raw[
        raw_feature_columns
    ].isna().sum().sum()
    == 0
)

assert (
    (
        construction_features_raw[
            raw_feature_columns
        ] < 0
    ).sum().sum()
    == 0
)

print("\nRAW CONSTRUCTION ROLLING FEATURE QC: PASS")

# ------------------------------------------------------------
# 9. Display a small audit sample
# ------------------------------------------------------------

audit_sample = construction_features_raw[
    construction_features_raw["ta_name"].isin(
        [
            "Auckland",
            "Far North District",
            "Christchurch City"
        ]
    )
][
    [
        "transition",
        "airbnb_anchor_date",
        "latest_available_consent_month",
        "ta_code",
        "ta_name",
        "dwelling_3m",
        "houses_3m",
        "apartments_3m",
        "retirement_3m",
        "townhouse_other_3m",
        "dwelling_12m",
        "houses_12m",
        "apartments_12m",
        "retirement_12m",
        "townhouse_other_12m"
    ]
].copy()

# Natural transition order for DISPLAY only
audit_sample["transition_number"] = (
    audit_sample["transition"]
    .str.replace("T", "", regex=False)
    .astype(int)
)

audit_sample = (
    audit_sample
    .sort_values(
        ["transition_number", "ta_name"]
    )
    .drop(columns="transition_number")
)

print("\nAUDIT SAMPLE")
display(audit_sample)

In [ ]:
# ============================================================
# STEP 21.3 — MANUAL SOURCE-TO-FEATURE REPRODUCIBILITY AUDIT
#
# Verify selected 3M and 12M dwelling totals directly against
# the underlying monthly Stats NZ records.
#
# No transformation.
# No feature modification.
# ============================================================

audit_cases = [
    ("T1",  "076", "Auckland"),
    ("T1",  "001", "Far North District"),
    ("T10", "076", "Auckland"),
    ("T10", "001", "Far North District")
]

manual_results = []

for transition, ta_code, expected_name in audit_cases:

    # --------------------------------------------------------
    # Locate feature row
    # --------------------------------------------------------

    feature_row = construction_features_raw[
        (construction_features_raw["transition"] == transition)
        &
        (construction_features_raw["ta_code"] == ta_code)
    ].iloc[0]

    anchor_period = feature_row[
        "latest_available_consent_period"
    ]

    print("\n" + "=" * 90)
    print(
        transition,
        "|",
        feature_row["ta_name"],
        "| latest available:",
        feature_row["latest_available_consent_month"]
    )
    print("=" * 90)

    assert feature_row["ta_name"] == expected_name

    # --------------------------------------------------------
    # Check 3M and 12M separately
    # --------------------------------------------------------

    for window in [3, 12]:

        start_period = anchor_period - (window - 1)

        source_rows = feature_source[
            (feature_source["ta_code"] == ta_code)
            &
            (feature_source["building_type"] == "Dwelling units")
            &
            (feature_source["consent_period"] >= start_period)
            &
            (feature_source["consent_period"] <= anchor_period)
        ][
            [
                "consent_month",
                "value"
            ]
        ].copy()

        source_rows = source_rows.sort_values(
            "consent_month"
        )

        manual_sum = source_rows["value"].sum()

        stored_value = feature_row[
            f"dwelling_{window}m"
        ]

        print(f"\n{window}-MONTH SOURCE RECORDS")
        display(source_rows)

        print("Manual source sum:", manual_sum)
        print(
            f"Stored dwelling_{window}m:",
            stored_value
        )
        print(
            "Difference:",
            manual_sum - stored_value
        )

        manual_results.append({
            "transition": transition,
            "ta_code": ta_code,
            "ta_name": expected_name,
            "window_months": window,
            "window_start":
                start_period.strftime("%YM%m"),
            "window_end":
                anchor_period.strftime("%YM%m"),
            "source_rows": len(source_rows),
            "manual_sum": manual_sum,
            "stored_value": stored_value,
            "difference": manual_sum - stored_value
        })

# ------------------------------------------------------------
# Final audit table
# ------------------------------------------------------------

manual_audit = pd.DataFrame(manual_results)

print("\n\nMANUAL REPRODUCIBILITY SUMMARY")
print("=" * 90)

display(manual_audit)

# ------------------------------------------------------------
# Hard QC
# ------------------------------------------------------------

assert len(manual_audit) == 8

assert (
    manual_audit["difference"] == 0
).all()

assert (
    manual_audit.loc[
        manual_audit["window_months"] == 3,
        "source_rows"
    ] == 3
).all()

assert (
    manual_audit.loc[
        manual_audit["window_months"] == 12,
        "source_rows"
    ] == 12
).all()

print("\n8/8 MANUAL SOURCE-TO-FEATURE CHECKS: PASS")
print("ALL DIFFERENCES: 0")
print("\nMANUAL REPRODUCIBILITY AUDIT: PASS")

## Step 21.4 — Add 6-month raw residential construction totals

This step adds a six-month trailing construction window to the raw feature layer.
The 3-month and 12-month windows remain the primary temporal specification,
while the 6-month window is retained for robustness analysis.

All windows terminate at the latest Building Consent month officially available
at each Airbnb forecast origin.

In [ ]:
# ============================================================
# STEP 21.4
# Add 6-month raw residential construction totals
# ============================================================

# Residential building types used in the construction feature block
residential_types = {
    "Dwelling units": "dwelling",
    "Houses": "houses",
    "Apartments": "apartments",
    "Retirement village units": "retirement",
    "Townhouses, flats, units, and other dwellings": "townhouse_other"
}

# Start from the already validated TA × transition backbone
construction_features_6m = construction_backbone.copy()

# ------------------------------------------------------------
# Calculate trailing 6-month totals
# ------------------------------------------------------------

for building_type, short_name in residential_types.items():

    values_6m = []

    for _, row in construction_features_6m.iterrows():

        ta_code = row["ta_code"]
        end_period = row["latest_available_consent_period"]

        # Inclusive six-month window:
        # e.g. latest = 2025M08
        #      start  = 2025M03
        start_period = end_period - 5

        source_rows = feature_source[
            (feature_source["ta_code"] == ta_code)
            & (feature_source["building_type"] == building_type)
            & (feature_source["consent_period"] >= start_period)
            & (feature_source["consent_period"] <= end_period)
        ].copy()

        # Each TA × transition × building type must have
        # exactly six monthly observations
        assert len(source_rows) == 6, (
            f"Expected 6 months but found {len(source_rows)} | "
            f"transition={row['transition']} | "
            f"TA={ta_code} | "
            f"type={building_type}"
        )

        values_6m.append(source_rows["value"].sum())

    construction_features_6m[f"{short_name}_6m"] = values_6m


# ------------------------------------------------------------
# Select the new 6M variables
# ------------------------------------------------------------

six_month_columns = [
    "transition",
    "ta_code",
    "dwelling_6m",
    "houses_6m",
    "apartments_6m",
    "retirement_6m",
    "townhouse_other_6m"
]


# ------------------------------------------------------------
# Protect against accidentally running this cell twice
# ------------------------------------------------------------

existing_6m = [
    col for col in six_month_columns[2:]
    if col in construction_features_raw.columns
]

if existing_6m:
    construction_features_raw = construction_features_raw.drop(
        columns=existing_6m
    )


# ------------------------------------------------------------
# Add 6M variables to the existing validated 3M + 12M table
# ------------------------------------------------------------

construction_features_raw = construction_features_raw.merge(
    construction_features_6m[six_month_columns],
    on=["transition", "ta_code"],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# Basic QC
# ------------------------------------------------------------

six_month_vars = [
    "dwelling_6m",
    "houses_6m",
    "apartments_6m",
    "retirement_6m",
    "townhouse_other_6m"
]

print("=" * 90)
print("STEP 21.4 — 6-MONTH RAW CONSTRUCTION FEATURES")
print("=" * 90)

print("Rows:", len(construction_features_raw))
print("Transitions:", construction_features_raw["transition"].nunique())
print("TA codes:", construction_features_raw["ta_code"].nunique())
print("6M variables:", len(six_month_vars))

print(
    "Missing 6M values:",
    construction_features_raw[six_month_vars]
    .isna()
    .sum()
    .sum()
)

print(
    "Negative 6M values:",
    (construction_features_raw[six_month_vars] < 0)
    .sum()
    .sum()
)

print(
    "Duplicate transition-TA:",
    construction_features_raw.duplicated(
        subset=["transition", "ta_code"]
    ).sum()
)


# ------------------------------------------------------------
# Residential accounting identity
#
# Dwelling units =
# Houses
# + Apartments
# + Retirement village units
# + Townhouses/flats/units/other dwellings
# ------------------------------------------------------------

component_sum_6m = (
    construction_features_raw["houses_6m"]
    + construction_features_raw["apartments_6m"]
    + construction_features_raw["retirement_6m"]
    + construction_features_raw["townhouse_other_6m"]
)

difference_6m = (
    construction_features_raw["dwelling_6m"]
    - component_sum_6m
)

print(
    "6M accounting mismatches:",
    (difference_6m != 0).sum()
)

print(
    "6M maximum absolute difference:",
    difference_6m.abs().max()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(construction_features_raw) == 670

assert (
    construction_features_raw["transition"].nunique()
    == 10
)

assert (
    construction_features_raw["ta_code"].nunique()
    == 67
)

assert (
    construction_features_raw[six_month_vars]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    (construction_features_raw[six_month_vars] < 0)
    .sum()
    .sum()
    == 0
)

assert (
    construction_features_raw.duplicated(
        subset=["transition", "ta_code"]
    ).sum()
    == 0
)

assert (difference_6m == 0).all()


# ------------------------------------------------------------
# Display selected cases for visual inspection
# ------------------------------------------------------------
audit_display = construction_features_raw[
    (
        construction_features_raw["transition"]
        .isin(["T1", "T10"])
    )
    &
    (
        construction_features_raw["ta_code"]
        .isin(["076", "001"])
    )
][
    [
        "transition",
        "ta_code",
        "ta_name",
        "latest_available_consent_month",
        "dwelling_3m",
        "dwelling_6m",
        "dwelling_12m"
    ]
].copy()

# Put T1 before T10 and Auckland/Far North in a readable order
transition_order = {"T1": 1, "T10": 10}

audit_display["transition_order"] = (
    audit_display["transition"].map(transition_order)
)

audit_display = (
    audit_display
    .sort_values(["transition_order", "ta_code"])
    .drop(columns="transition_order")
)

display(audit_display)

print("\n6-MONTH RAW CONSTRUCTION FEATURE QC: PASS")

## Step 21.5 — Full 3M/6M/12M consistency QC

This step verifies the internal consistency of all residential rolling-window
construction totals across all 67 Territorial Authorities and 10 forecast
origins.

Because the rolling windows are nested and monthly consent counts are
non-negative, each construction measure should satisfy:

3M <= 6M <= 12M.

Residential accounting identities are also rechecked for all three windows.

In [ ]:
# ============================================================
# STEP 21.5
# Full 3M / 6M / 12M consistency QC
# ============================================================

component_names = [
    "dwelling",
    "houses",
    "apartments",
    "retirement",
    "townhouse_other"
]

windows = [3, 6, 12]

print("=" * 90)
print("STEP 21.5 — FULL 3M / 6M / 12M CONSISTENCY QC")
print("=" * 90)

# ------------------------------------------------------------
# 1. Structural integrity
# ------------------------------------------------------------

print("\n1. STRUCTURAL INTEGRITY")
print("-" * 90)

print("Rows:", len(construction_features_raw))
print("Transitions:", construction_features_raw["transition"].nunique())
print("TA codes:", construction_features_raw["ta_code"].nunique())

duplicate_count = construction_features_raw.duplicated(
    subset=["transition", "ta_code"]
).sum()

print("Duplicate transition-TA:", duplicate_count)

assert len(construction_features_raw) == 670
assert construction_features_raw["transition"].nunique() == 10
assert construction_features_raw["ta_code"].nunique() == 67
assert duplicate_count == 0


# ------------------------------------------------------------
# 2. Missing and negative values across all rolling variables
# ------------------------------------------------------------

all_rolling_vars = [
    f"{component}_{window}m"
    for component in component_names
    for window in windows
]

missing_count = (
    construction_features_raw[all_rolling_vars]
    .isna()
    .sum()
    .sum()
)

negative_count = (
    construction_features_raw[all_rolling_vars] < 0
).sum().sum()

print("\n2. VALUE INTEGRITY")
print("-" * 90)

print("Rolling variables checked:", len(all_rolling_vars))
print("Missing values:", missing_count)
print("Negative values:", negative_count)

assert missing_count == 0
assert negative_count == 0


# ------------------------------------------------------------
# 3. Residential accounting identity for each window
# ------------------------------------------------------------

print("\n3. RESIDENTIAL ACCOUNTING IDENTITY")
print("-" * 90)

accounting_results = []

for window in windows:

    component_sum = (
        construction_features_raw[f"houses_{window}m"]
        + construction_features_raw[f"apartments_{window}m"]
        + construction_features_raw[f"retirement_{window}m"]
        + construction_features_raw[f"townhouse_other_{window}m"]
    )

    difference = (
        construction_features_raw[f"dwelling_{window}m"]
        - component_sum
    )

    mismatch_count = (difference != 0).sum()
    max_abs_difference = difference.abs().max()

    accounting_results.append({
        "window": f"{window}M",
        "mismatches": mismatch_count,
        "max_absolute_difference": max_abs_difference
    })

accounting_qc = pd.DataFrame(accounting_results)

display(accounting_qc)

assert (accounting_qc["mismatches"] == 0).all()
assert (accounting_qc["max_absolute_difference"] == 0).all()


# ------------------------------------------------------------
# 4. Nested-window consistency
#
# Since monthly counts are non-negative and:
#
# 3M is contained in 6M
# 6M is contained in 12M
#
# we require:
#
# 3M <= 6M <= 12M
# ------------------------------------------------------------

print("\n4. NESTED-WINDOW CONSISTENCY")
print("-" * 90)

nesting_results = []

for component in component_names:

    violation_3_6 = (
        construction_features_raw[f"{component}_3m"]
        >
        construction_features_raw[f"{component}_6m"]
    )

    violation_6_12 = (
        construction_features_raw[f"{component}_6m"]
        >
        construction_features_raw[f"{component}_12m"]
    )

    nesting_results.append({
        "component": component,
        "3M_gt_6M": int(violation_3_6.sum()),
        "6M_gt_12M": int(violation_6_12.sum()),
        "total_violations": int(
            violation_3_6.sum() + violation_6_12.sum()
        )
    })

nesting_qc = pd.DataFrame(nesting_results)

display(nesting_qc)

assert (nesting_qc["3M_gt_6M"] == 0).all()
assert (nesting_qc["6M_gt_12M"] == 0).all()


# ------------------------------------------------------------
# 5. Summary distributions for dwelling totals
# ------------------------------------------------------------

print("\n5. DWELLING WINDOW SUMMARY")
print("-" * 90)

dwelling_summary = construction_features_raw[
    ["dwelling_3m", "dwelling_6m", "dwelling_12m"]
].describe().T[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(dwelling_summary)


# ------------------------------------------------------------
# 6. Zero counts
# Important for later ratios / shares
# ------------------------------------------------------------

print("\n6. ZERO COUNTS")
print("-" * 90)

zero_results = []

for component in component_names:

    row = {"component": component}

    for window in windows:
        col = f"{component}_{window}m"
        row[f"zero_{window}m"] = int(
            (construction_features_raw[col] == 0).sum()
        )

    zero_results.append(row)

zero_qc = pd.DataFrame(zero_results)

display(zero_qc)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FULL 3M / 6M / 12M CONSTRUCTION CONSISTENCY QC: PASS")
print("=" * 90)

## Step 21.6 — Zero-denominator audit before feature transformation

This step identifies TA-transition observations with zero dwelling consents
in the 12-month reference window.

A zero 12-month dwelling total makes dwelling-based momentum and residential
composition ratios mathematically undefined. These cases are inspected before
any ratio or share features are constructed.

In [ ]:
# ============================================================
# STEP 21.6
# Zero-denominator audit before feature transformation
# ============================================================

print("=" * 90)
print("STEP 21.6 — ZERO-DENOMINATOR AUDIT")
print("=" * 90)


# ------------------------------------------------------------
# 1. Identify zero dwelling totals by temporal window
# ------------------------------------------------------------

zero_dwelling_summary = pd.DataFrame({
    "window": ["3M", "6M", "12M"],
    "zero_dwelling_rows": [
        int((construction_features_raw["dwelling_3m"] == 0).sum()),
        int((construction_features_raw["dwelling_6m"] == 0).sum()),
        int((construction_features_raw["dwelling_12m"] == 0).sum())
    ]
})

print("\n1. ZERO DWELLING TOTALS BY WINDOW")
print("-" * 90)

display(zero_dwelling_summary)


# ------------------------------------------------------------
# 2. Identify the exact 12M zero-denominator case(s)
#
# IMPORTANT:
# Keep latest_available_consent_period because this is the
# validated pandas monthly Period object already used by the
# rolling-window construction pipeline.
# ------------------------------------------------------------

zero_12m_cases = construction_features_raw[
    construction_features_raw["dwelling_12m"] == 0
][
    [
        "transition",
        "ta_code",
        "ta_name",
        "latest_available_consent_month",
        "latest_available_consent_period",
        "dwelling_3m",
        "dwelling_6m",
        "dwelling_12m",
        "houses_12m",
        "apartments_12m",
        "retirement_12m",
        "townhouse_other_12m"
    ]
].copy()

print("\n2. EXACT 12M ZERO-DENOMINATOR CASE(S)")
print("-" * 90)

display(zero_12m_cases)

print("Number of 12M zero cases:", len(zero_12m_cases))


# ------------------------------------------------------------
# 3. Verify nested-window zero logic
#
# Because:
#
#     3M subset 6M subset 12M
#
# and all monthly dwelling counts are non-negative,
# if dwelling_12m = 0, then dwelling_6m and dwelling_3m
# must also equal zero.
# ------------------------------------------------------------

zero_12m_logic_failures = zero_12m_cases[
    (zero_12m_cases["dwelling_3m"] != 0)
    |
    (zero_12m_cases["dwelling_6m"] != 0)
]

print("\n3. NESTED-WINDOW ZERO LOGIC")
print("-" * 90)

print(
    "12M-zero cases with non-zero 3M or 6M:",
    len(zero_12m_logic_failures)
)

assert len(zero_12m_logic_failures) == 0


# ------------------------------------------------------------
# 4. Inspect the underlying 12 monthly Dwelling-unit records
#
# DO NOT parse latest_available_consent_month again.
#
# We use latest_available_consent_period directly because it is
# already a validated pandas Period object with monthly frequency.
# ------------------------------------------------------------

source_audit_rows = []

for _, row in zero_12m_cases.iterrows():

    transition = row["transition"]
    ta_code = row["ta_code"]
    ta_name = row["ta_name"]

    # Validated Period object from the existing pipeline
    end_period = row["latest_available_consent_period"]

    # Inclusive 12-month window:
    # end month plus the preceding 11 months
    start_period = end_period - 11

    source_rows = feature_source[
        (feature_source["ta_code"] == ta_code)
        & (feature_source["building_type"] == "Dwelling units")
        & (feature_source["consent_period"] >= start_period)
        & (feature_source["consent_period"] <= end_period)
    ][
        [
            "consent_month",
            "ta_code",
            "ta_name",
            "building_type",
            "value"
        ]
    ].copy()

    source_rows = source_rows.sort_values("consent_month")

    print("\n" + "=" * 90)
    print(
        transition,
        "|",
        ta_code,
        "|",
        ta_name,
        "| 12M window:",
        start_period.strftime("%YM%m"),
        "to",
        end_period.strftime("%YM%m")
    )
    print("=" * 90)

    display(source_rows)

    print("Source rows:", len(source_rows))
    print("12M source sum:", source_rows["value"].sum())
    print("Stored dwelling_12m:", row["dwelling_12m"])

    source_audit_rows.append({
        "transition": transition,
        "ta_code": ta_code,
        "ta_name": ta_name,
        "window_start": start_period.strftime("%YM%m"),
        "window_end": end_period.strftime("%YM%m"),
        "source_rows": len(source_rows),
        "source_sum": source_rows["value"].sum(),
        "stored_dwelling_12m": row["dwelling_12m"]
    })


# ------------------------------------------------------------
# 5. Summarise source audit
# ------------------------------------------------------------

zero_12m_source_audit = pd.DataFrame(source_audit_rows)

print("\n4. ZERO-CASE SOURCE AUDIT SUMMARY")
print("-" * 90)

display(zero_12m_source_audit)


# ------------------------------------------------------------
# 6. Assertions
#
# Every identified 12M zero case must:
# - contain exactly 12 source months
# - have a source sum of zero
# - match the stored 12M feature exactly
# ------------------------------------------------------------

assert len(zero_12m_source_audit) == len(zero_12m_cases)

assert (
    zero_12m_source_audit["source_rows"] == 12
).all()

assert (
    zero_12m_source_audit["source_sum"] == 0
).all()

assert (
    zero_12m_source_audit["stored_dwelling_12m"] == 0
).all()


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ZERO-DENOMINATOR SOURCE AUDIT: PASS")
print("=" * 90)

## Step 21.7A — Construction magnitude features

This step derives log-transformed construction-magnitude features from the
validated dwelling-consent rolling totals.

The primary specification retains the 3-month and 12-month windows.
The 6-month transformation is also retained for later robustness analysis.

The transformation uses log1p(x) = log(1 + x), which is defined at zero
and reduces the influence of the strongly right-skewed distribution of
Territorial Authority consent counts without modifying the underlying
raw consent values.

In [ ]:
# ============================================================
# STEP 21.7A
# Construction magnitude features
# ============================================================

import numpy as np

print("=" * 90)
print("STEP 21.7A — CONSTRUCTION MAGNITUDE FEATURES")
print("=" * 90)


# ------------------------------------------------------------
# 1. Preserve raw dwelling totals
# ------------------------------------------------------------

raw_magnitude_cols = [
    "dwelling_3m",
    "dwelling_6m",
    "dwelling_12m"
]

assert construction_features_raw[raw_magnitude_cols].isna().sum().sum() == 0
assert (construction_features_raw[raw_magnitude_cols] < 0).sum().sum() == 0


# ------------------------------------------------------------
# 2. Create log1p magnitude features
#
# 3M and 12M = primary temporal specification
# 6M         = retained for robustness analysis
# ------------------------------------------------------------

construction_features_raw["construction_dwelling_3m_log"] = np.log1p(
    construction_features_raw["dwelling_3m"]
)

construction_features_raw["construction_dwelling_6m_log"] = np.log1p(
    construction_features_raw["dwelling_6m"]
)

construction_features_raw["construction_dwelling_12m_log"] = np.log1p(
    construction_features_raw["dwelling_12m"]
)


# ------------------------------------------------------------
# 3. QC: missing / infinite / negative transformed values
# ------------------------------------------------------------

log_cols = [
    "construction_dwelling_3m_log",
    "construction_dwelling_6m_log",
    "construction_dwelling_12m_log"
]

missing_log = construction_features_raw[log_cols].isna().sum().sum()

infinite_log = np.isinf(
    construction_features_raw[log_cols].to_numpy()
).sum()

negative_log = (
    construction_features_raw[log_cols] < 0
).sum().sum()

print("\n1. TRANSFORMATION INTEGRITY")
print("-" * 90)

print("Log features created:", len(log_cols))
print("Missing transformed values:", missing_log)
print("Infinite transformed values:", infinite_log)
print("Negative transformed values:", negative_log)

assert missing_log == 0
assert infinite_log == 0
assert negative_log == 0


# ------------------------------------------------------------
# 4. Verify exact mathematical transformation
# ------------------------------------------------------------

check_3m = np.allclose(
    construction_features_raw["construction_dwelling_3m_log"],
    np.log1p(construction_features_raw["dwelling_3m"])
)

check_6m = np.allclose(
    construction_features_raw["construction_dwelling_6m_log"],
    np.log1p(construction_features_raw["dwelling_6m"])
)

check_12m = np.allclose(
    construction_features_raw["construction_dwelling_12m_log"],
    np.log1p(construction_features_raw["dwelling_12m"])
)

print("\n2. EXACT TRANSFORMATION CHECK")
print("-" * 90)

print("3M log1p exact:", check_3m)
print("6M log1p exact:", check_6m)
print("12M log1p exact:", check_12m)

assert check_3m
assert check_6m
assert check_12m


# ------------------------------------------------------------
# 5. Verify zero preservation
#
# Raw zero must transform exactly to log1p(0) = 0.
# ------------------------------------------------------------

zero_check_results = []

for window in [3, 6, 12]:

    raw_col = f"dwelling_{window}m"
    log_col = f"construction_dwelling_{window}m_log"

    raw_zero = construction_features_raw[raw_col] == 0

    zero_count = int(raw_zero.sum())

    transformed_zero_count = int(
        (
            construction_features_raw.loc[raw_zero, log_col]
            == 0
        ).sum()
    )

    zero_check_results.append({
        "window": f"{window}M",
        "raw_zero_rows": zero_count,
        "correctly_transformed_to_zero": transformed_zero_count
    })

zero_transformation_qc = pd.DataFrame(zero_check_results)

print("\n3. ZERO-PRESERVATION CHECK")
print("-" * 90)

display(zero_transformation_qc)

assert (
    zero_transformation_qc["raw_zero_rows"]
    ==
    zero_transformation_qc["correctly_transformed_to_zero"]
).all()


# ------------------------------------------------------------
# 6. Compare raw and transformed distributions
# ------------------------------------------------------------

print("\n4. RAW VS LOG DISTRIBUTION SUMMARY")
print("-" * 90)

distribution_rows = []

for window in [3, 6, 12]:

    raw_col = f"dwelling_{window}m"
    log_col = f"construction_dwelling_{window}m_log"

    distribution_rows.append({
        "window": f"{window}M",
        "raw_min": construction_features_raw[raw_col].min(),
        "raw_median": construction_features_raw[raw_col].median(),
        "raw_mean": construction_features_raw[raw_col].mean(),
        "raw_max": construction_features_raw[raw_col].max(),
        "log_min": construction_features_raw[log_col].min(),
        "log_median": construction_features_raw[log_col].median(),
        "log_mean": construction_features_raw[log_col].mean(),
        "log_max": construction_features_raw[log_col].max()
    })

magnitude_distribution_qc = pd.DataFrame(distribution_rows)

display(magnitude_distribution_qc)


# ------------------------------------------------------------
# 7. Inspect selected real cases
# ------------------------------------------------------------

selected_magnitude_cases = construction_features_raw[
    (
        construction_features_raw["transition"].isin(["T1", "T10"])
    )
    &
    (
        construction_features_raw["ta_code"].isin(["001", "067", "076"])
    )
][
    [
        "transition",
        "ta_code",
        "ta_name",
        "dwelling_3m",
        "construction_dwelling_3m_log",
        "dwelling_6m",
        "construction_dwelling_6m_log",
        "dwelling_12m",
        "construction_dwelling_12m_log"
    ]
].copy()

display(selected_magnitude_cases)


# ------------------------------------------------------------
# 8. Explicitly verify Chatham T10
# ------------------------------------------------------------

chatham_t10 = construction_features_raw[
    (construction_features_raw["transition"] == "T10")
    &
    (construction_features_raw["ta_code"] == "067")
]

assert len(chatham_t10) == 1

assert chatham_t10["dwelling_3m"].iloc[0] == 0
assert chatham_t10["dwelling_6m"].iloc[0] == 0
assert chatham_t10["dwelling_12m"].iloc[0] == 0

assert chatham_t10["construction_dwelling_3m_log"].iloc[0] == 0
assert chatham_t10["construction_dwelling_6m_log"].iloc[0] == 0
assert chatham_t10["construction_dwelling_12m_log"].iloc[0] == 0

print("\n5. CHATHAM T10 ZERO CHECK")
print("-" * 90)
print("Raw 3M / 6M / 12M: 0 / 0 / 0")
print("Log 3M / 6M / 12M: 0 / 0 / 0")
print("Chatham T10 retained without imputation: PASS")


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CONSTRUCTION MAGNITUDE FEATURE QC: PASS")
print("=" * 90)

## Step 21.7B-1 — Construction momentum definition audit

This diagnostic step evaluates two candidate definitions of construction momentum
before the final feature definition is frozen.

Candidate A compares the recent 3-month monthly consent pace with the average
monthly pace over the full trailing 12-month period.

Candidate B compares the recent 3-month monthly consent pace with the immediately
preceding 9-month monthly pace. The two comparison periods in Candidate B do not
overlap.

The candidates are evaluated for mathematical validity and interpretability,
not for predictive performance.

In [ ]:
# ============================================================
# STEP 21.7B-1
# Construction momentum definition audit
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("STEP 21.7B-1 — CONSTRUCTION MOMENTUM DEFINITION AUDIT")
print("=" * 90)


# ------------------------------------------------------------
# 1. Construct the underlying monthly pace measures
# ------------------------------------------------------------

momentum_audit = construction_features_raw[
    [
        "transition",
        "ta_code",
        "ta_name",
        "latest_available_consent_month",
        "dwelling_3m",
        "dwelling_12m"
    ]
].copy()


# Recent 3-month average monthly pace
momentum_audit["recent_3m_monthly_pace"] = (
    momentum_audit["dwelling_3m"] / 3.0
)


# Previous 9 months:
#
# 12M = previous 9M + recent 3M
#
# Therefore:
# previous_9m = 12M - 3M
momentum_audit["prior_9m_total"] = (
    momentum_audit["dwelling_12m"]
    - momentum_audit["dwelling_3m"]
)


# Previous 9-month average monthly pace
momentum_audit["prior_9m_monthly_pace"] = (
    momentum_audit["prior_9m_total"] / 9.0
)


# Full 12-month average monthly pace
momentum_audit["annual_12m_monthly_pace"] = (
    momentum_audit["dwelling_12m"] / 12.0
)


# ------------------------------------------------------------
# 2. Basic mathematical QC before calculating momentum
# ------------------------------------------------------------

print("\n1. UNDERLYING PACE QC")
print("-" * 90)

negative_prior9 = (
    momentum_audit["prior_9m_total"] < 0
).sum()

missing_paces = momentum_audit[
    [
        "recent_3m_monthly_pace",
        "prior_9m_total",
        "prior_9m_monthly_pace",
        "annual_12m_monthly_pace"
    ]
].isna().sum().sum()

print("Rows:", len(momentum_audit))
print("Negative prior-9M totals:", negative_prior9)
print("Missing underlying pace values:", missing_paces)

assert len(momentum_audit) == 670
assert negative_prior9 == 0
assert missing_paces == 0


# ------------------------------------------------------------
# 3. Candidate A
#
# Recent 3M monthly pace
# versus
# full trailing 12M monthly pace
#
# NOTE:
# The 12M reference contains the recent 3M period.
# ------------------------------------------------------------

momentum_audit["momentum_A_3m_vs_12m"] = (
    np.log1p(momentum_audit["recent_3m_monthly_pace"])
    -
    np.log1p(momentum_audit["annual_12m_monthly_pace"])
)


# ------------------------------------------------------------
# 4. Candidate B
#
# Recent 3M monthly pace
# versus
# immediately preceding 9M monthly pace
#
# These periods do NOT overlap.
# ------------------------------------------------------------

momentum_audit["momentum_B_3m_vs_prior9m"] = (
    np.log1p(momentum_audit["recent_3m_monthly_pace"])
    -
    np.log1p(momentum_audit["prior_9m_monthly_pace"])
)


# ------------------------------------------------------------
# 5. Mathematical integrity of both candidates
# ------------------------------------------------------------

candidate_cols = [
    "momentum_A_3m_vs_12m",
    "momentum_B_3m_vs_prior9m"
]

missing_momentum = (
    momentum_audit[candidate_cols]
    .isna()
    .sum()
    .sum()
)

infinite_momentum = np.isinf(
    momentum_audit[candidate_cols].to_numpy()
).sum()

print("\n2. MOMENTUM MATHEMATICAL INTEGRITY")
print("-" * 90)

print("Missing momentum values:", missing_momentum)
print("Infinite momentum values:", infinite_momentum)

assert missing_momentum == 0
assert infinite_momentum == 0


# ------------------------------------------------------------
# 6. Check sign interpretation
#
# For Candidate B:
#
# positive -> recent 3M pace > prior 9M pace
# zero     -> equal pace
# negative -> recent 3M pace < prior 9M pace
# ------------------------------------------------------------

tolerance = 1e-12

expected_positive = (
    momentum_audit["recent_3m_monthly_pace"]
    >
    momentum_audit["prior_9m_monthly_pace"]
)

expected_negative = (
    momentum_audit["recent_3m_monthly_pace"]
    <
    momentum_audit["prior_9m_monthly_pace"]
)

expected_equal = np.isclose(
    momentum_audit["recent_3m_monthly_pace"],
    momentum_audit["prior_9m_monthly_pace"],
    atol=tolerance,
    rtol=0
)

actual_positive = (
    momentum_audit["momentum_B_3m_vs_prior9m"] > tolerance
)

actual_negative = (
    momentum_audit["momentum_B_3m_vs_prior9m"] < -tolerance
)

actual_equal = np.isclose(
    momentum_audit["momentum_B_3m_vs_prior9m"],
    0,
    atol=tolerance,
    rtol=0
)

positive_mismatches = (
    expected_positive != actual_positive
).sum()

negative_mismatches = (
    expected_negative != actual_negative
).sum()

equal_mismatches = (
    expected_equal != actual_equal
).sum()

print("\n3. CANDIDATE B SIGN-INTERPRETATION QC")
print("-" * 90)

print("Positive-sign mismatches:", positive_mismatches)
print("Negative-sign mismatches:", negative_mismatches)
print("Zero-sign mismatches:", equal_mismatches)

assert positive_mismatches == 0
assert negative_mismatches == 0
assert equal_mismatches == 0


# ------------------------------------------------------------
# 7. Distribution of the two candidate momentum measures
# ------------------------------------------------------------

print("\n4. MOMENTUM DISTRIBUTION SUMMARY")
print("-" * 90)

momentum_summary = momentum_audit[
    candidate_cols
].describe().T[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(momentum_summary)


# ------------------------------------------------------------
# 8. Count increasing / stable / decreasing cases
#    using Candidate B
# ------------------------------------------------------------

conditions = [
    momentum_audit["momentum_B_3m_vs_prior9m"] > tolerance,
    momentum_audit["momentum_B_3m_vs_prior9m"] < -tolerance
]

choices = [
    "recent pace higher",
    "recent pace lower"
]

momentum_audit["momentum_B_direction"] = np.select(
    conditions,
    choices,
    default="equal pace"
)

print("\n5. CANDIDATE B DIRECTION COUNTS")
print("-" * 90)

direction_counts = (
    momentum_audit["momentum_B_direction"]
    .value_counts()
    .rename_axis("direction")
    .reset_index(name="rows")
)

display(direction_counts)


# ------------------------------------------------------------
# 9. Inspect selected cases:
#    Far North, Auckland, Chatham
#    at T1 and T10 where available
# ------------------------------------------------------------

print("\n6. SELECTED CASES")
print("-" * 90)

selected_cases = momentum_audit[
    (
        momentum_audit["transition"].isin(["T1", "T10"])
    )
    &
    (
        momentum_audit["ta_code"].isin(["001", "067", "076"])
    )
][
    [
        "transition",
        "ta_code",
        "ta_name",
        "dwelling_3m",
        "dwelling_12m",
        "recent_3m_monthly_pace",
        "prior_9m_total",
        "prior_9m_monthly_pace",
        "annual_12m_monthly_pace",
        "momentum_A_3m_vs_12m",
        "momentum_B_3m_vs_prior9m",
        "momentum_B_direction"
    ]
].copy()

transition_order = {
    "T1": 1,
    "T10": 10
}

selected_cases["transition_order"] = (
    selected_cases["transition"].map(transition_order)
)

selected_cases = (
    selected_cases
    .sort_values(["transition_order", "ta_code"])
    .drop(columns="transition_order")
)

display(selected_cases)


# ------------------------------------------------------------
# 10. Explicit Chatham T10 zero-case audit
# ------------------------------------------------------------

print("\n7. CHATHAM T10 MOMENTUM CHECK")
print("-" * 90)

chatham_t10_momentum = momentum_audit[
    (momentum_audit["transition"] == "T10")
    &
    (momentum_audit["ta_code"] == "067")
]

assert len(chatham_t10_momentum) == 1

chatham_row = chatham_t10_momentum.iloc[0]

print(
    "Recent 3M monthly pace:",
    chatham_row["recent_3m_monthly_pace"]
)

print(
    "Prior 9M monthly pace:",
    chatham_row["prior_9m_monthly_pace"]
)

print(
    "12M monthly pace:",
    chatham_row["annual_12m_monthly_pace"]
)

print(
    "Candidate A:",
    chatham_row["momentum_A_3m_vs_12m"]
)

print(
    "Candidate B:",
    chatham_row["momentum_B_3m_vs_prior9m"]
)

print(
    "Candidate B direction:",
    chatham_row["momentum_B_direction"]
)

assert chatham_row["recent_3m_monthly_pace"] == 0
assert chatham_row["prior_9m_monthly_pace"] == 0
assert chatham_row["annual_12m_monthly_pace"] == 0

assert np.isclose(
    chatham_row["momentum_A_3m_vs_12m"],
    0
)

assert np.isclose(
    chatham_row["momentum_B_3m_vs_prior9m"],
    0
)

print("Chatham T10 handled without imputation: PASS")


# ------------------------------------------------------------
# 11. Find strongest increases and decreases under Candidate B
#
# Diagnostic only.
# We are NOT selecting a definition based on these values.
# ------------------------------------------------------------

print("\n8. STRONGEST CANDIDATE B INCREASES")
print("-" * 90)

strongest_increases = momentum_audit.nlargest(
    10,
    "momentum_B_3m_vs_prior9m"
)[
    [
        "transition",
        "ta_code",
        "ta_name",
        "recent_3m_monthly_pace",
        "prior_9m_monthly_pace",
        "momentum_B_3m_vs_prior9m"
    ]
]

display(strongest_increases)


print("\n9. STRONGEST CANDIDATE B DECREASES")
print("-" * 90)

strongest_decreases = momentum_audit.nsmallest(
    10,
    "momentum_B_3m_vs_prior9m"
)[
    [
        "transition",
        "ta_code",
        "ta_name",
        "recent_3m_monthly_pace",
        "prior_9m_monthly_pace",
        "momentum_B_3m_vs_prior9m"
    ]
]

display(strongest_decreases)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MOMENTUM DEFINITION AUDIT: PASS")
print("=" * 90)

print(
    "\nNOTE: No final momentum definition has been frozen yet."
)

print(
    "Candidate A = recent 3M pace versus overlapping 12M pace."
)

print(
    "Candidate B = recent 3M pace versus non-overlapping prior 9M pace."
)

## Step 21.7B-2 — Freeze the construction momentum feature

Construction momentum is defined as the log difference between the average
monthly dwelling-consent pace in the most recent three months and the average
monthly pace in the immediately preceding nine months.

The comparison periods are non-overlapping. Positive values indicate a higher
recent consent pace, negative values indicate a lower recent pace, and zero
indicates equal observed rates. A zero value arising when both periods contain
zero dwelling consents is retained as a valid mathematical result and is
distinguished later through the no-dwelling-activity indicator.

In [ ]:
# ============================================================
# STEP 21.7B-2
# Freeze official construction momentum feature
# ============================================================

import numpy as np

print("=" * 90)
print("STEP 21.7B-2 — FREEZE CONSTRUCTION MOMENTUM FEATURE")
print("=" * 90)


# ------------------------------------------------------------
# 1. Construct non-overlapping recent and prior periods
# ------------------------------------------------------------

construction_features_raw["dwelling_prior9m"] = (
    construction_features_raw["dwelling_12m"]
    - construction_features_raw["dwelling_3m"]
)

construction_features_raw["dwelling_recent3m_monthly_pace"] = (
    construction_features_raw["dwelling_3m"] / 3.0
)

construction_features_raw["dwelling_prior9m_monthly_pace"] = (
    construction_features_raw["dwelling_prior9m"] / 9.0
)


# ------------------------------------------------------------
# 2. Official momentum feature
# ------------------------------------------------------------

construction_features_raw[
    "construction_momentum_3m_vs_prior9m"
] = (
    np.log1p(
        construction_features_raw[
            "dwelling_recent3m_monthly_pace"
        ]
    )
    -
    np.log1p(
        construction_features_raw[
            "dwelling_prior9m_monthly_pace"
        ]
    )
)


# ------------------------------------------------------------
# 3. Mathematical integrity
# ------------------------------------------------------------

momentum_col = "construction_momentum_3m_vs_prior9m"

missing_count = (
    construction_features_raw[momentum_col]
    .isna()
    .sum()
)

infinite_count = np.isinf(
    construction_features_raw[momentum_col].to_numpy()
).sum()

negative_prior9 = (
    construction_features_raw["dwelling_prior9m"] < 0
).sum()

print("\n1. MATHEMATICAL INTEGRITY")
print("-" * 90)

print("Rows:", len(construction_features_raw))
print("Negative prior-9M totals:", negative_prior9)
print("Missing momentum values:", missing_count)
print("Infinite momentum values:", infinite_count)

assert len(construction_features_raw) == 670
assert negative_prior9 == 0
assert missing_count == 0
assert infinite_count == 0


# ------------------------------------------------------------
# 4. Exact reproduction of audited Candidate B
# ------------------------------------------------------------

expected_momentum = (
    np.log1p(
        construction_features_raw["dwelling_3m"] / 3.0
    )
    -
    np.log1p(
        (
            construction_features_raw["dwelling_12m"]
            - construction_features_raw["dwelling_3m"]
        ) / 9.0
    )
)

exact_match = np.allclose(
    construction_features_raw[momentum_col],
    expected_momentum
)

print("\n2. EXACT FORMULA CHECK")
print("-" * 90)

print("Official momentum reproduces Candidate B:", exact_match)

assert exact_match


# ------------------------------------------------------------
# 5. Sign interpretation
# ------------------------------------------------------------

recent = construction_features_raw[
    "dwelling_recent3m_monthly_pace"
]

prior = construction_features_raw[
    "dwelling_prior9m_monthly_pace"
]

momentum = construction_features_raw[momentum_col]

tol = 1e-12

positive_mismatch = (
    (recent > prior)
    !=
    (momentum > tol)
).sum()

negative_mismatch = (
    (recent < prior)
    !=
    (momentum < -tol)
).sum()

equal_mismatch = (
    np.isclose(recent, prior, atol=tol, rtol=0)
    !=
    np.isclose(momentum, 0, atol=tol, rtol=0)
).sum()

print("\n3. SIGN INTERPRETATION")
print("-" * 90)

print("Positive-sign mismatches:", positive_mismatch)
print("Negative-sign mismatches:", negative_mismatch)
print("Zero-sign mismatches:", equal_mismatch)

assert positive_mismatch == 0
assert negative_mismatch == 0
assert equal_mismatch == 0


# ------------------------------------------------------------
# 6. Distribution
# ------------------------------------------------------------

print("\n4. OFFICIAL MOMENTUM DISTRIBUTION")
print("-" * 90)

display(
    construction_features_raw[
        [momentum_col]
    ].describe().T
)


# ------------------------------------------------------------
# 7. Explicit Chatham T10 check
# ------------------------------------------------------------

chatham = construction_features_raw[
    (construction_features_raw["transition"] == "T10")
    &
    (construction_features_raw["ta_code"] == "067")
]

assert len(chatham) == 1

print("\n5. CHATHAM T10")
print("-" * 90)

display(
    chatham[
        [
            "transition",
            "ta_code",
            "ta_name",
            "dwelling_3m",
            "dwelling_prior9m",
            "dwelling_recent3m_monthly_pace",
            "dwelling_prior9m_monthly_pace",
            momentum_col
        ]
    ]
)

assert chatham["dwelling_3m"].iloc[0] == 0
assert chatham["dwelling_prior9m"].iloc[0] == 0
assert np.isclose(chatham[momentum_col].iloc[0], 0)

print("Chatham zero-zero case retained correctly: PASS")


print("\n" + "=" * 90)
print("OFFICIAL CONSTRUCTION MOMENTUM FEATURE: PASS")
print("=" * 90)

## Step 21.7C-1 — Residential construction composition audit

This diagnostic step evaluates whether the composition of the trailing
12-month residential construction pipeline provides a sufficiently stable
and interpretable feature for the primary construction specification.

Composition is evaluated using the four mutually exhaustive dwelling
categories reported by Stats NZ: houses, apartments, retirement village
units, and townhouses/flats/units/other dwellings.

The audit examines:
1. the accounting identity between total dwellings and dwelling subtypes;
2. the mathematical validity of subtype shares;
3. undefined composition when total dwelling consents equal zero;
4. the empirical distribution and sparsity of candidate composition measures;
5. the sensitivity of extreme composition shares to small denominators; and
6. the respective contributions of apartments and townhouses/other dwellings
   to a candidate combined multi-unit measure.

No composition feature is frozen at this stage. The audit is used to determine
whether a composition feature is sufficiently interpretable and stable for
subsequent implementation.

In [ ]:
# ============================================================
# STEP 21.7C-1
# Residential construction composition audit
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 21.7C-1 — RESIDENTIAL CONSTRUCTION COMPOSITION AUDIT")
print("=" * 100)


# ============================================================
# 0. Build a diagnostic copy only
#    DO NOT modify the frozen construction feature table yet.
# ============================================================

required_cols = [
    "transition",
    "ta_code",
    "ta_name",
    "dwelling_12m",
    "houses_12m",
    "apartments_12m",
    "retirement_12m",
    "townhouse_other_12m"
]

missing_required = [
    c for c in required_cols
    if c not in construction_features_raw.columns
]

assert len(missing_required) == 0, (
    f"Missing required columns: {missing_required}"
)

composition_audit = construction_features_raw[
    required_cols
].copy()


# ============================================================
# 1. Structural integrity
# ============================================================

print("\n1. STRUCTURAL INTEGRITY")
print("-" * 100)

print("Rows:", len(composition_audit))
print(
    "Transitions:",
    composition_audit["transition"].nunique()
)
print(
    "TA codes:",
    composition_audit["ta_code"].nunique()
)

duplicate_keys = composition_audit.duplicated(
    subset=["transition", "ta_code"]
).sum()

raw_missing = composition_audit[
    [
        "dwelling_12m",
        "houses_12m",
        "apartments_12m",
        "retirement_12m",
        "townhouse_other_12m"
    ]
].isna().sum().sum()

raw_negative = (
    composition_audit[
        [
            "dwelling_12m",
            "houses_12m",
            "apartments_12m",
            "retirement_12m",
            "townhouse_other_12m"
        ]
    ] < 0
).sum().sum()

print("Duplicate transition-TA keys:", duplicate_keys)
print("Missing raw 12M values:", raw_missing)
print("Negative raw 12M values:", raw_negative)

assert len(composition_audit) == 670
assert composition_audit["transition"].nunique() == 10
assert composition_audit["ta_code"].nunique() == 67
assert duplicate_keys == 0
assert raw_missing == 0
assert raw_negative == 0

print("Structural integrity: PASS")


# ============================================================
# 2. Re-confirm residential accounting identity
#
# D = H + A + R + T
# ============================================================

composition_audit["component_sum_12m"] = (
    composition_audit["houses_12m"]
    + composition_audit["apartments_12m"]
    + composition_audit["retirement_12m"]
    + composition_audit["townhouse_other_12m"]
)

composition_audit["accounting_difference"] = (
    composition_audit["dwelling_12m"]
    - composition_audit["component_sum_12m"]
)

accounting_mismatches = (
    composition_audit["accounting_difference"] != 0
).sum()

max_accounting_difference = (
    composition_audit["accounting_difference"]
    .abs()
    .max()
)

print("\n2. 12-MONTH RESIDENTIAL ACCOUNTING IDENTITY")
print("-" * 100)

print("Accounting mismatches:", accounting_mismatches)
print(
    "Maximum absolute accounting difference:",
    max_accounting_difference
)

assert accounting_mismatches == 0
assert max_accounting_difference == 0

print("D = H + A + R + T: PASS")


# ============================================================
# 3. Identify valid and undefined denominators
# ============================================================

positive_denominator = (
    composition_audit["dwelling_12m"] > 0
)

zero_denominator = (
    composition_audit["dwelling_12m"] == 0
)

n_positive = positive_denominator.sum()
n_zero = zero_denominator.sum()

print("\n3. DENOMINATOR AUDIT")
print("-" * 100)

print("D12 > 0:", n_positive)
print("D12 = 0:", n_zero)
print(
    "Percent D12 = 0:",
    round(100 * n_zero / len(composition_audit), 4),
    "%"
)

assert n_positive + n_zero == len(composition_audit)

print("\nZero-denominator cases:")

display(
    composition_audit.loc[
        zero_denominator,
        [
            "transition",
            "ta_code",
            "ta_name",
            "dwelling_12m",
            "houses_12m",
            "apartments_12m",
            "retirement_12m",
            "townhouse_other_12m"
        ]
    ]
)


# ============================================================
# 4. Create diagnostic composition shares
#
# IMPORTANT:
# Shares are defined ONLY where D12 > 0.
# D12 = 0 remains NaN.
#
# No epsilon.
# No pseudo-count.
# No fillna(0).
# ============================================================

share_definitions = {
    "house_share_12m":
        "houses_12m",

    "apartment_share_12m":
        "apartments_12m",

    "retirement_share_12m":
        "retirement_12m",

    "townhouse_other_share_12m":
        "townhouse_other_12m"
}

for share_col, numerator_col in share_definitions.items():

    composition_audit[share_col] = np.where(
        positive_denominator,
        composition_audit[numerator_col]
        / composition_audit["dwelling_12m"],
        np.nan
    )


# Candidate combined composition measure
composition_audit["multiunit_count_12m"] = (
    composition_audit["apartments_12m"]
    + composition_audit["townhouse_other_12m"]
)

composition_audit["multiunit_share_12m"] = np.where(
    positive_denominator,
    composition_audit["multiunit_count_12m"]
    / composition_audit["dwelling_12m"],
    np.nan
)


share_cols = [
    "house_share_12m",
    "apartment_share_12m",
    "retirement_share_12m",
    "townhouse_other_share_12m",
    "multiunit_share_12m"
]


# ============================================================
# 5. Mathematical validity of shares
# ============================================================

print("\n4. SHARE MATHEMATICAL VALIDITY")
print("-" * 100)

valid_shares = composition_audit.loc[
    positive_denominator,
    share_cols
]

below_zero = (valid_shares < 0).sum().sum()
above_one = (valid_shares > 1).sum().sum()
missing_when_positive = valid_shares.isna().sum().sum()

undefined_share_count = (
    composition_audit.loc[
        zero_denominator,
        share_cols
    ].isna().sum().sum()
)

expected_undefined = n_zero * len(share_cols)

print("Shares < 0:", below_zero)
print("Shares > 1:", above_one)
print(
    "Missing shares when D12 > 0:",
    missing_when_positive
)
print(
    "Undefined shares when D12 = 0:",
    undefined_share_count,
    "/",
    expected_undefined
)

assert below_zero == 0
assert above_one == 0
assert missing_when_positive == 0
assert undefined_share_count == expected_undefined

print("Share range and missingness behaviour: PASS")


# ============================================================
# 6. Four-component share identity
#
# H_share + A_share + R_share + T_share = 1
# ============================================================

composition_audit["four_share_sum"] = (
    composition_audit["house_share_12m"]
    + composition_audit["apartment_share_12m"]
    + composition_audit["retirement_share_12m"]
    + composition_audit["townhouse_other_share_12m"]
)

share_sum_error = (
    composition_audit.loc[
        positive_denominator,
        "four_share_sum"
    ] - 1.0
).abs()

share_sum_mismatches = (
    share_sum_error > 1e-12
).sum()

max_share_sum_error = share_sum_error.max()

print("\n5. FOUR-COMPONENT SHARE IDENTITY")
print("-" * 100)

print(
    "Share-sum mismatches:",
    share_sum_mismatches
)
print(
    "Maximum absolute error from 1:",
    max_share_sum_error
)

assert share_sum_mismatches == 0

print("H + A + R + T shares = 1: PASS")


# ============================================================
# 7. Multi-unit identity
#
# multiunit_share = apartment_share + townhouse_other_share
# ============================================================

multiunit_identity_error = (
    composition_audit.loc[
        positive_denominator,
        "multiunit_share_12m"
    ]
    -
    (
        composition_audit.loc[
            positive_denominator,
            "apartment_share_12m"
        ]
        +
        composition_audit.loc[
            positive_denominator,
            "townhouse_other_share_12m"
        ]
    )
).abs()

multiunit_identity_mismatches = (
    multiunit_identity_error > 1e-12
).sum()

print("\n6. MULTI-UNIT IDENTITY")
print("-" * 100)

print(
    "Identity mismatches:",
    multiunit_identity_mismatches
)
print(
    "Maximum absolute difference:",
    multiunit_identity_error.max()
)

assert multiunit_identity_mismatches == 0

print(
    "multiunit_share = apartment_share + "
    "townhouse_other_share: PASS"
)


# ============================================================
# 8. Distribution and sparsity
# ============================================================

print("\n7. COMPOSITION DISTRIBUTION")
print("-" * 100)

composition_summary = (
    composition_audit.loc[
        positive_denominator,
        share_cols
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

display(composition_summary)


print("\n8. ZERO / ONE / INTERIOR SHARE COUNTS")
print("-" * 100)

sparsity_rows = []

for col in share_cols:

    s = composition_audit.loc[
        positive_denominator,
        col
    ]

    n = len(s)

    zeros = np.isclose(
        s,
        0,
        atol=1e-12,
        rtol=0
    ).sum()

    ones = np.isclose(
        s,
        1,
        atol=1e-12,
        rtol=0
    ).sum()

    interior = n - zeros - ones

    sparsity_rows.append(
        {
            "feature": col,
            "valid_n": n,
            "zero_n": zeros,
            "zero_pct": 100 * zeros / n,
            "one_n": ones,
            "one_pct": 100 * ones / n,
            "interior_n": interior,
            "interior_pct": 100 * interior / n
        }
    )

composition_sparsity = pd.DataFrame(
    sparsity_rows
)

display(composition_sparsity)


# ============================================================
# 9. Multi-unit denominator sensitivity
#
# We need to know whether extreme shares are mainly generated
# by very small total dwelling counts.
# ============================================================

print("\n9. MULTI-UNIT SHARE VS DENOMINATOR SIZE")
print("-" * 100)

valid_multiunit = composition_audit.loc[
    positive_denominator,
    [
        "transition",
        "ta_code",
        "ta_name",
        "dwelling_12m",
        "apartments_12m",
        "townhouse_other_12m",
        "multiunit_count_12m",
        "multiunit_share_12m"
    ]
].copy()


# Descriptive denominator bands ONLY.
# These are not model thresholds.
valid_multiunit["denominator_band"] = pd.cut(
    valid_multiunit["dwelling_12m"],
    bins=[
        -np.inf,
        5,
        10,
        25,
        50,
        100,
        250,
        500,
        np.inf
    ],
    labels=[
        "1-5",
        "6-10",
        "11-25",
        "26-50",
        "51-100",
        "101-250",
        "251-500",
        ">500"
    ]
)

denominator_summary = (
    valid_multiunit
    .groupby(
        "denominator_band",
        observed=False
    )
    .agg(
        observations=(
            "multiunit_share_12m",
            "size"
        ),
        median_dwelling_12m=(
            "dwelling_12m",
            "median"
        ),
        median_multiunit_share=(
            "multiunit_share_12m",
            "median"
        ),
        mean_multiunit_share=(
            "multiunit_share_12m",
            "mean"
        ),
        share_zero_pct=(
            "multiunit_share_12m",
            lambda x:
                100 * np.isclose(
                    x,
                    0,
                    atol=1e-12,
                    rtol=0
                ).mean()
        ),
        share_one_pct=(
            "multiunit_share_12m",
            lambda x:
                100 * np.isclose(
                    x,
                    1,
                    atol=1e-12,
                    rtol=0
                ).mean()
        )
    )
    .reset_index()
)

display(denominator_summary)


# ============================================================
# 10. Inspect extreme multi-unit shares
# ============================================================

print("\n10. HIGHEST MULTI-UNIT SHARES")
print("-" * 100)

highest_multiunit = (
    valid_multiunit
    .sort_values(
        [
            "multiunit_share_12m",
            "dwelling_12m"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(15)
)

display(highest_multiunit)


print("\n11. LOWEST NON-ZERO MULTI-UNIT SHARES")
print("-" * 100)

lowest_nonzero_multiunit = (
    valid_multiunit[
        valid_multiunit[
            "multiunit_share_12m"
        ] > 0
    ]
    .sort_values(
        [
            "multiunit_share_12m",
            "dwelling_12m"
        ],
        ascending=[
            True,
            True
        ]
    )
    .head(15)
)

display(lowest_nonzero_multiunit)


# ============================================================
# 11. Audit high shares specifically
#
# These thresholds are diagnostic only.
# They are NOT feature-engineering cut-offs.
# ============================================================

print("\n12. HIGH-SHARE DENOMINATOR AUDIT")
print("-" * 100)

high_share_thresholds = [
    0.50,
    0.75,
    0.90,
    1.00
]

high_share_rows = []

for threshold in high_share_thresholds:

    if threshold < 1.0:

        mask = (
            valid_multiunit[
                "multiunit_share_12m"
            ] >= threshold
        )

        label = f">={threshold:.2f}"

    else:

        mask = np.isclose(
            valid_multiunit[
                "multiunit_share_12m"
            ],
            1.0,
            atol=1e-12,
            rtol=0
        )

        label = "=1.00"

    subset = valid_multiunit.loc[mask]

    high_share_rows.append(
        {
            "share_threshold": label,
            "observations": len(subset),
            "median_dwelling_12m":
                subset["dwelling_12m"].median()
                if len(subset) > 0
                else np.nan,
            "min_dwelling_12m":
                subset["dwelling_12m"].min()
                if len(subset) > 0
                else np.nan,
            "max_dwelling_12m":
                subset["dwelling_12m"].max()
                if len(subset) > 0
                else np.nan,
            "pct_with_D12_le_10":
                100 * (
                    subset["dwelling_12m"] <= 10
                ).mean()
                if len(subset) > 0
                else np.nan,
            "pct_with_D12_le_25":
                100 * (
                    subset["dwelling_12m"] <= 25
                ).mean()
                if len(subset) > 0
                else np.nan
        }
    )

high_share_audit = pd.DataFrame(
    high_share_rows
)

display(high_share_audit)


# ============================================================
# 12. Apartment vs townhouse contribution
#
# Important because "multi-unit" combines two source categories.
# ============================================================

print("\n13. APARTMENT VS TOWNHOUSE/OTHER CONTRIBUTION")
print("-" * 100)

positive_multiunit = (
    composition_audit["multiunit_count_12m"] > 0
)

composition_audit[
    "apartment_fraction_of_multiunit"
] = np.where(
    positive_multiunit,
    composition_audit["apartments_12m"]
    / composition_audit["multiunit_count_12m"],
    np.nan
)

composition_audit[
    "townhouse_fraction_of_multiunit"
] = np.where(
    positive_multiunit,
    composition_audit["townhouse_other_12m"]
    / composition_audit["multiunit_count_12m"],
    np.nan
)

contribution_summary = (
    composition_audit.loc[
        positive_multiunit,
        [
            "apartment_fraction_of_multiunit",
            "townhouse_fraction_of_multiunit"
        ]
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95
        ]
    )
    .T
)

display(contribution_summary)


# ============================================================
# 13. Selected substantive cases
#
# Same cases used in earlier construction QC for continuity.
# ============================================================

print("\n14. SELECTED CASES")
print("-" * 100)

selected_cases = composition_audit[
    (
        composition_audit[
            "transition"
        ].isin(["T1", "T10"])
    )
    &
    (
        composition_audit[
            "ta_code"
        ].isin(["001", "067", "076"])
    )
][
    [
        "transition",
        "ta_code",
        "ta_name",
        "dwelling_12m",
        "houses_12m",
        "apartments_12m",
        "townhouse_other_12m",
        "retirement_12m",
        "house_share_12m",
        "apartment_share_12m",
        "townhouse_other_share_12m",
        "retirement_share_12m",
        "multiunit_share_12m"
    ]
].copy()

transition_order = {
    "T1": 1,
    "T10": 10
}

selected_cases["transition_order"] = (
    selected_cases["transition"]
    .map(transition_order)
)

selected_cases = (
    selected_cases
    .sort_values(
        [
            "transition_order",
            "ta_code"
        ]
    )
    .drop(
        columns="transition_order"
    )
)

display(selected_cases)


# ============================================================
# 14. Explicit Chatham T10 undefined-composition check
# ============================================================

print("\n15. CHATHAM T10 UNDEFINED-COMPOSITION CHECK")
print("-" * 100)

chatham_t10 = composition_audit[
    (
        composition_audit["transition"] == "T10"
    )
    &
    (
        composition_audit["ta_code"] == "067"
    )
]

assert len(chatham_t10) == 1

chatham = chatham_t10.iloc[0]

print(
    "Dwelling 12M:",
    chatham["dwelling_12m"]
)
print(
    "Multi-unit count 12M:",
    chatham["multiunit_count_12m"]
)
print(
    "Multi-unit share 12M:",
    chatham["multiunit_share_12m"]
)

assert chatham["dwelling_12m"] == 0
assert chatham["multiunit_count_12m"] == 0
assert pd.isna(
    chatham["multiunit_share_12m"]
)

print(
    "Chatham composition retained as undefined (NaN): PASS"
)


# ============================================================
# 15. Check deterministic redundancy explicitly
# ============================================================

print("\n16. DETERMINISTIC REDUNDANCY CHECK")
print("-" * 100)

valid = composition_audit.loc[
    positive_denominator
].copy()

three_group_sum = (
    valid["house_share_12m"]
    + valid["multiunit_share_12m"]
    + valid["retirement_share_12m"]
)

three_group_error = (
    three_group_sum - 1.0
).abs()

three_group_mismatches = (
    three_group_error > 1e-12
).sum()

print(
    "house_share + multiunit_share + "
    "retirement_share != 1:",
    three_group_mismatches
)

assert three_group_mismatches == 0

print(
    "Confirmed: including all three would create "
    "deterministic compositional redundancy."
)


# ============================================================
# 16. Final diagnostic status
# ============================================================

print("\n" + "=" * 100)
print("RESIDENTIAL CONSTRUCTION COMPOSITION AUDIT: PASS")
print("=" * 100)

print(
    "\nIMPORTANT:"
)
print(
    "- No composition feature has been frozen."
)
print(
    "- No zero denominator has been replaced by zero."
)
print(
    "- No pseudo-count or epsilon has been introduced."
)
print(
    "- No denominator threshold has been used to alter the data."
)
print(
    "- Denominator bands and high-share thresholds above are "
    "diagnostic only."
)

## Step 21.7C-2 — Freeze official residential construction composition feature

Following the diagnostic composition audit, one 12-month residential
composition measure is retained for the primary construction specification.

The official feature is:

construction_multiunit_share_12m
= (apartments_12m + townhouse_other_12m) / dwelling_12m

where the denominator is positive.

The feature represents the proportion of dwelling units consented during
the trailing 12-month point-in-time window that were classified as
apartments or townhouses/flats/units/other dwellings.

When dwelling_12m = 0, composition is mathematically undefined and the
feature is retained as missing (NaN), rather than being replaced with zero,
an epsilon, or a pseudo-count.

A companion indicator, no_dwelling_12m, identifies observations with no
dwelling units consented during the 12-month window.

No denominator threshold or smoothing transformation is applied.

In [ ]:
# ============================================================
# STEP 21.7C-2
# Freeze official residential construction composition feature
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 21.7C-2 — FREEZE OFFICIAL CONSTRUCTION COMPOSITION FEATURE")
print("=" * 100)


# ============================================================
# 0. Required source columns
# ============================================================

required_cols = [
    "transition",
    "ta_code",
    "dwelling_12m",
    "apartments_12m",
    "townhouse_other_12m"
]

missing_required = [
    c for c in required_cols
    if c not in construction_features_raw.columns
]

assert len(missing_required) == 0, (
    f"Missing required columns: {missing_required}"
)

print("\nRequired source columns: PASS")


# ============================================================
# 1. Protect against accidental reruns
#
# If this cell is rerun, remove only the official composition
# columns created by this step before recreating them.
# ============================================================

official_cols = [
    "construction_multiunit_share_12m",
    "no_dwelling_12m"
]

existing_official_cols = [
    c for c in official_cols
    if c in construction_features_raw.columns
]

if existing_official_cols:

    print(
        "\nExisting official composition columns detected "
        "from a previous run:"
    )
    print(existing_official_cols)

    construction_features_raw = (
        construction_features_raw
        .drop(columns=existing_official_cols)
        .copy()
    )

    print(
        "Existing columns removed before exact recreation."
    )

else:

    print(
        "\nNo previous official composition columns detected."
    )


# ============================================================
# 2. Re-confirm denominator conditions before construction
# ============================================================

positive_denominator = (
    construction_features_raw["dwelling_12m"] > 0
)

zero_denominator = (
    construction_features_raw["dwelling_12m"] == 0
)

negative_denominator = (
    construction_features_raw["dwelling_12m"] < 0
)

n_positive = int(positive_denominator.sum())
n_zero = int(zero_denominator.sum())
n_negative = int(negative_denominator.sum())

print("\n1. DENOMINATOR STATUS")
print("-" * 100)

print("D12 > 0:", n_positive)
print("D12 = 0:", n_zero)
print("D12 < 0:", n_negative)

assert n_negative == 0
assert n_positive + n_zero == len(construction_features_raw)

print("Denominator status: PASS")


# ============================================================
# 3. Create official composition feature
#
# IMPORTANT:
# - no epsilon
# - no pseudo-count
# - no denominator threshold
# - no fillna(0)
# ============================================================

multiunit_count_12m = (
    construction_features_raw["apartments_12m"]
    + construction_features_raw["townhouse_other_12m"]
)

construction_features_raw[
    "construction_multiunit_share_12m"
] = np.where(
    positive_denominator,
    multiunit_count_12m
    / construction_features_raw["dwelling_12m"],
    np.nan
)


# Companion indicator:
#
# 0 = at least one dwelling consent exists in the 12M window
# 1 = no dwelling consent exists; composition is undefined

construction_features_raw[
    "no_dwelling_12m"
] = zero_denominator.astype("int8")


print("\nOfficial composition columns created.")


# ============================================================
# 4. Structural QC after feature creation
# ============================================================

print("\n2. STRUCTURAL QC")
print("-" * 100)

print(
    "Rows:",
    len(construction_features_raw)
)

print(
    "Transitions:",
    construction_features_raw[
        "transition"
    ].nunique()
)

print(
    "TA codes:",
    construction_features_raw[
        "ta_code"
    ].nunique()
)

duplicate_keys = (
    construction_features_raw
    .duplicated(
        subset=["transition", "ta_code"]
    )
    .sum()
)

print(
    "Duplicate transition-TA keys:",
    duplicate_keys
)

assert len(construction_features_raw) == 670
assert (
    construction_features_raw[
        "transition"
    ].nunique()
    == 10
)
assert (
    construction_features_raw[
        "ta_code"
    ].nunique()
    == 67
)
assert duplicate_keys == 0

print("Structural QC: PASS")


# ============================================================
# 5. Exact formula reproduction
# ============================================================

expected_share = np.where(
    positive_denominator,
    (
        construction_features_raw["apartments_12m"]
        + construction_features_raw["townhouse_other_12m"]
    )
    / construction_features_raw["dwelling_12m"],
    np.nan
)

actual_share = construction_features_raw[
    "construction_multiunit_share_12m"
]


valid_mask = positive_denominator

formula_difference = np.abs(
    actual_share.loc[valid_mask].to_numpy()
    -
    expected_share[valid_mask.to_numpy()]
)

formula_mismatches = int(
    (formula_difference > 1e-12).sum()
)

max_formula_difference = (
    formula_difference.max()
    if len(formula_difference) > 0
    else np.nan
)

print("\n3. EXACT FORMULA REPRODUCTION")
print("-" * 100)

print(
    "Formula mismatches:",
    formula_mismatches
)

print(
    "Maximum absolute difference:",
    max_formula_difference
)

assert formula_mismatches == 0

print("Exact formula reproduction: PASS")


# ============================================================
# 6. Range QC
# ============================================================

valid_share = construction_features_raw.loc[
    positive_denominator,
    "construction_multiunit_share_12m"
]

below_zero = int(
    (valid_share < 0).sum()
)

above_one = int(
    (valid_share > 1).sum()
)

infinite_values = int(
    np.isinf(valid_share).sum()
)

print("\n4. RANGE QC")
print("-" * 100)

print("Shares < 0:", below_zero)
print("Shares > 1:", above_one)
print("Infinite shares:", infinite_values)

print(
    "Minimum valid share:",
    valid_share.min()
)

print(
    "Maximum valid share:",
    valid_share.max()
)

assert below_zero == 0
assert above_one == 0
assert infinite_values == 0

print("Share range QC: PASS")


# ============================================================
# 7. Missingness semantics
#
# NaN is allowed if and only if D12 = 0.
# ============================================================

share_is_missing = (
    construction_features_raw[
        "construction_multiunit_share_12m"
    ].isna()
)

missing_share_count = int(
    share_is_missing.sum()
)

missing_when_positive = int(
    (
        share_is_missing
        &
        positive_denominator
    ).sum()
)

defined_when_zero = int(
    (
        ~share_is_missing
        &
        zero_denominator
    ).sum()
)

print("\n5. MISSINGNESS SEMANTICS")
print("-" * 100)

print(
    "Total missing composition shares:",
    missing_share_count
)

print(
    "Missing when D12 > 0:",
    missing_when_positive
)

print(
    "Defined when D12 = 0:",
    defined_when_zero
)

assert missing_share_count == n_zero
assert missing_when_positive == 0
assert defined_when_zero == 0

print(
    "NaN occurs if and only if D12 = 0: PASS"
)


# ============================================================
# 8. no_dwelling_12m indicator QC
# ============================================================

print("\n6. NO-DWELLING INDICATOR QC")
print("-" * 100)

indicator_values = sorted(
    construction_features_raw[
        "no_dwelling_12m"
    ].unique()
    .tolist()
)

indicator_positive_count = int(
    construction_features_raw[
        "no_dwelling_12m"
    ].sum()
)

indicator_mismatch = int(
    (
        construction_features_raw[
            "no_dwelling_12m"
        ].astype(bool)
        != zero_denominator
    ).sum()
)

print(
    "Unique indicator values:",
    indicator_values
)

print(
    "no_dwelling_12m = 1:",
    indicator_positive_count
)

print(
    "Indicator/denominator mismatches:",
    indicator_mismatch
)

assert set(indicator_values).issubset({0, 1})
assert indicator_positive_count == n_zero
assert indicator_mismatch == 0

print(
    "no_dwelling_12m indicator: PASS"
)


# ============================================================
# 9. Distinguish:
#
# A. observed zero composition
# B. undefined composition
#
# These must NOT be treated as the same state.
# ============================================================

observed_zero_composition = (
    positive_denominator
    &
    np.isclose(
        construction_features_raw[
            "construction_multiunit_share_12m"
        ],
        0.0,
        atol=1e-12,
        rtol=0
    )
)

undefined_composition = (
    zero_denominator
    &
    construction_features_raw[
        "construction_multiunit_share_12m"
    ].isna()
)

n_observed_zero = int(
    observed_zero_composition.sum()
)

n_undefined = int(
    undefined_composition.sum()
)

print("\n7. ZERO VS UNDEFINED COMPOSITION")
print("-" * 100)

print(
    "Observed zero composition "
    "(D12 > 0 and share = 0):",
    n_observed_zero
)

print(
    "Undefined composition "
    "(D12 = 0 and share = NaN):",
    n_undefined
)

assert n_observed_zero > 0
assert n_undefined == n_zero

print(
    "Observed zero and undefined composition "
    "remain distinct: PASS"
)


# ============================================================
# 10. Distribution confirmation
# ============================================================

print("\n8. OFFICIAL FEATURE DISTRIBUTION")
print("-" * 100)

official_distribution = (
    valid_share
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

display(
    official_distribution.to_frame(
        name="construction_multiunit_share_12m"
    )
)


# ============================================================
# 11. Explicit Chatham T10 check
# ============================================================

print("\n9. CHATHAM T10 CHECK")
print("-" * 100)

chatham_t10 = construction_features_raw[
    (
        construction_features_raw[
            "transition"
        ] == "T10"
    )
    &
    (
        construction_features_raw[
            "ta_code"
        ] == "067"
    )
]

assert len(chatham_t10) == 1

chatham = chatham_t10.iloc[0]

print(
    "Dwelling 12M:",
    chatham["dwelling_12m"]
)

print(
    "Apartment 12M:",
    chatham["apartments_12m"]
)

print(
    "Townhouse/other 12M:",
    chatham["townhouse_other_12m"]
)

print(
    "Official multi-unit share:",
    chatham[
        "construction_multiunit_share_12m"
    ]
)

print(
    "no_dwelling_12m:",
    chatham[
        "no_dwelling_12m"
    ]
)

assert chatham["dwelling_12m"] == 0

assert pd.isna(
    chatham[
        "construction_multiunit_share_12m"
    ]
)

assert (
    chatham[
        "no_dwelling_12m"
    ]
    == 1
)

print(
    "Chatham T10 undefined composition "
    "retained correctly: PASS"
)


# ============================================================
# 12. Selected cases for substantive verification
# ============================================================

print("\n10. SELECTED CASE VERIFICATION")
print("-" * 100)

selected_cases = (
    construction_features_raw[
        (
            construction_features_raw[
                "transition"
            ].isin(["T1", "T10"])
        )
        &
        (
            construction_features_raw[
                "ta_code"
            ].isin(
                ["001", "067", "076"]
            )
        )
    ][
        [
            "transition",
            "ta_code",
            "dwelling_12m",
            "apartments_12m",
            "townhouse_other_12m",
            "construction_multiunit_share_12m",
            "no_dwelling_12m"
        ]
    ]
    .copy()
)

selected_cases[
    "_transition_order"
] = (
    selected_cases[
        "transition"
    ]
    .str.extract(
        r"(\d+)",
        expand=False
    )
    .astype(int)
)

selected_cases = (
    selected_cases
    .sort_values(
        [
            "_transition_order",
            "ta_code"
        ]
    )
    .drop(
        columns="_transition_order"
    )
)

display(selected_cases)


# ============================================================
# 13. Final frozen-feature inventory
# ============================================================

print("\n11. CONSTRUCTION FEATURE SPECIFICATION")
print("-" * 100)

primary_features = [
    "construction_dwelling_3m_log",
    "construction_dwelling_12m_log",
    "construction_momentum_3m_vs_prior9m",
    "construction_multiunit_share_12m"
]

robustness_features = [
    "construction_dwelling_6m_log"
]

technical_indicator = [
    "no_dwelling_12m"
]

all_expected = (
    primary_features
    + robustness_features
    + technical_indicator
)

missing_final_features = [
    c for c in all_expected
    if c not in construction_features_raw.columns
]

print("PRIMARY:")
for c in primary_features:
    print("  -", c)

print("\nROBUSTNESS:")
for c in robustness_features:
    print("  -", c)

print("\nMISSINGNESS / TECHNICAL INDICATOR:")
for c in technical_indicator:
    print("  -", c)

print(
    "\nMissing expected feature columns:",
    missing_final_features
)

assert len(missing_final_features) == 0


# ============================================================
# 14. Final status
# ============================================================

print("\n" + "=" * 100)
print(
    "OFFICIAL CONSTRUCTION COMPOSITION FEATURE: PASS"
)
print("=" * 100)

print(
    "\nFROZEN PRIMARY COMPOSITION FEATURE:"
)
print(
    "construction_multiunit_share_12m"
)

print(
    "\nDefinition:"
)
print(
    "(apartments_12m + townhouse_other_12m) "
    "/ dwelling_12m, where dwelling_12m > 0"
)

print(
    "\nZero-denominator rule:"
)
print(
    "dwelling_12m = 0 -> "
    "construction_multiunit_share_12m = NaN "
    "and no_dwelling_12m = 1"
)

print(
    "\nNo epsilon, pseudo-count, denominator threshold, "
    "or zero imputation has been applied."
)

print(
    "\nSTEP 21.7C-2 STATUS: FROZEN"
)

## Step 21.8 — Build and freeze final TA × transition construction feature table

This step creates the frozen construction feature table used for subsequent
multimodal integration.

The table retains:
- transition and point-in-time temporal provenance;
- Territorial Authority identifiers;
- the four pre-specified primary construction features;
- the six-month magnitude feature for robustness analysis; and
- the no-dwelling indicator required to preserve the distinction between
  observed zero composition and undefined composition.

Raw and diagnostic construction variables remain available in the upstream
processed data but are not included as primary model predictors.

The final table is validated before saving and is then reloaded from disk
to verify persistence without alteration.

In [ ]:
# ============================================================
# STEP 21.8
# Build, save, reload and freeze final construction feature table
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 21.8 — BUILD AND FREEZE FINAL CONSTRUCTION FEATURE TABLE")
print("=" * 100)


# ============================================================
# 0. Define the frozen specification
# ============================================================

id_cols = [
    "transition",
    "ta_code"
]

# Keep temporal provenance if already present in the validated
# construction feature table.
candidate_provenance_cols = [
    "anchor_date",
    "latest_available_consent_period",
    "release_date",
    "release_lag_days"
]

provenance_cols = [
    c for c in candidate_provenance_cols
    if c in construction_features_raw.columns
]

primary_features = [
    "construction_dwelling_3m_log",
    "construction_dwelling_12m_log",
    "construction_momentum_3m_vs_prior9m",
    "construction_multiunit_share_12m"
]

robustness_features = [
    "construction_dwelling_6m_log"
]

technical_cols = [
    "no_dwelling_12m"
]

final_cols = (
    id_cols
    + provenance_cols
    + primary_features
    + robustness_features
    + technical_cols
)

missing_required = [
    c for c in (
        id_cols
        + primary_features
        + robustness_features
        + technical_cols
    )
    if c not in construction_features_raw.columns
]

assert len(missing_required) == 0, (
    f"Missing required columns: {missing_required}"
)

print("\n1. FROZEN COLUMN SPECIFICATION")
print("-" * 100)

print("Identifier columns:")
for c in id_cols:
    print("  -", c)

print("\nTemporal provenance columns found:")
if provenance_cols:
    for c in provenance_cols:
        print("  -", c)
else:
    print("  - None retained from current table")

print("\nPrimary construction features:")
for c in primary_features:
    print("  -", c)

print("\nRobustness feature:")
for c in robustness_features:
    print("  -", c)

print("\nTechnical indicator:")
for c in technical_cols:
    print("  -", c)


# ============================================================
# 1. Build final table
# ============================================================

construction_features_final = (
    construction_features_raw[
        final_cols
    ]
    .copy()
)

print("\nFinal construction table created.")


# ============================================================
# 2. Structural integrity
# ============================================================

print("\n2. STRUCTURAL INTEGRITY")
print("-" * 100)

n_rows = len(construction_features_final)

n_transitions = (
    construction_features_final[
        "transition"
    ].nunique()
)

n_tas = (
    construction_features_final[
        "ta_code"
    ].nunique()
)

duplicate_keys = (
    construction_features_final
    .duplicated(
        subset=[
            "transition",
            "ta_code"
        ]
    )
    .sum()
)

print("Rows:", n_rows)
print("Transitions:", n_transitions)
print("TA codes:", n_tas)
print("Duplicate transition-TA keys:", duplicate_keys)

assert n_rows == 670
assert n_transitions == 10
assert n_tas == 67
assert duplicate_keys == 0

print("Structural integrity: PASS")


# ============================================================
# 3. Confirm exactly 67 TAs per transition
# ============================================================

print("\n3. TRANSITION COVERAGE")
print("-" * 100)

transition_coverage = (
    construction_features_final
    .groupby("transition")
    .agg(
        rows=("ta_code", "size"),
        unique_tas=("ta_code", "nunique")
    )
    .reset_index()
)

transition_coverage["_order"] = (
    transition_coverage[
        "transition"
    ]
    .str.extract(
        r"(\d+)",
        expand=False
    )
    .astype(int)
)

transition_coverage = (
    transition_coverage
    .sort_values("_order")
    .drop(columns="_order")
)

display(transition_coverage)

assert (
    transition_coverage["rows"] == 67
).all()

assert (
    transition_coverage["unique_tas"] == 67
).all()

print("67 TAs per transition: PASS")


# ============================================================
# 4. TA999 must NOT appear in this table
#
# This table represents construction-context-eligible TAs.
# TA999 will be handled as unavailable at listing-level merge.
# ============================================================

print("\n4. TA999 EXCLUSION CHECK")
print("-" * 100)

ta999_count = (
    construction_features_final[
        "ta_code"
    ]
    .astype(str)
    .eq("999")
    .sum()
)

print("TA999 rows:", ta999_count)

assert ta999_count == 0

print(
    "TA999 correctly excluded from eligible construction feature table: PASS"
)


# ============================================================
# 5. Primary feature missingness
# ============================================================

print("\n5. PRIMARY FEATURE MISSINGNESS")
print("-" * 100)

primary_missingness = pd.DataFrame({
    "feature": primary_features,
    "missing_n": [
        construction_features_final[c]
        .isna()
        .sum()
        for c in primary_features
    ]
})

primary_missingness["missing_pct"] = (
    100
    * primary_missingness["missing_n"]
    / len(construction_features_final)
)

display(primary_missingness)


# Expected:
# first 3 primary features = no missing
# composition = exactly one missing value

assert (
    construction_features_final[
        "construction_dwelling_3m_log"
    ].isna().sum()
    == 0
)

assert (
    construction_features_final[
        "construction_dwelling_12m_log"
    ].isna().sum()
    == 0
)

assert (
    construction_features_final[
        "construction_momentum_3m_vs_prior9m"
    ].isna().sum()
    == 0
)

assert (
    construction_features_final[
        "construction_multiunit_share_12m"
    ].isna().sum()
    == 1
)

print("Primary feature missingness: PASS")


# ============================================================
# 6. Robustness + technical column QC
# ============================================================

print("\n6. ROBUSTNESS / TECHNICAL QC")
print("-" * 100)

robustness_missing = (
    construction_features_final[
        "construction_dwelling_6m_log"
    ].isna().sum()
)

indicator_missing = (
    construction_features_final[
        "no_dwelling_12m"
    ].isna().sum()
)

indicator_positive = int(
    construction_features_final[
        "no_dwelling_12m"
    ].sum()
)

print(
    "Missing construction_dwelling_6m_log:",
    robustness_missing
)

print(
    "Missing no_dwelling_12m:",
    indicator_missing
)

print(
    "no_dwelling_12m = 1:",
    indicator_positive
)

assert robustness_missing == 0
assert indicator_missing == 0
assert indicator_positive == 1

print("Robustness / technical QC: PASS")


# ============================================================
# 7. Infinite-value QC
# ============================================================

print("\n7. INFINITE VALUE QC")
print("-" * 100)

numeric_feature_cols = (
    primary_features
    + robustness_features
)

infinite_counts = {}

for c in numeric_feature_cols:

    values = (
        construction_features_final[c]
        .dropna()
        .to_numpy(dtype=float)
    )

    infinite_counts[c] = int(
        np.isinf(values).sum()
    )

infinite_table = pd.DataFrame(
    {
        "feature": list(infinite_counts.keys()),
        "infinite_n": list(infinite_counts.values())
    }
)

display(infinite_table)

assert sum(infinite_counts.values()) == 0

print("Infinite-value QC: PASS")


# ============================================================
# 8. Re-confirm zero vs undefined composition semantics
# ============================================================

print("\n8. COMPOSITION SEMANTICS")
print("-" * 100)

composition_missing = (
    construction_features_final[
        "construction_multiunit_share_12m"
    ].isna()
)

no_dwelling = (
    construction_features_final[
        "no_dwelling_12m"
    ] == 1
)

semantic_mismatches = int(
    (
        composition_missing
        != no_dwelling
    ).sum()
)

print(
    "Composition/indicator semantic mismatches:",
    semantic_mismatches
)

assert semantic_mismatches == 0

print(
    "Undefined composition corresponds exactly "
    "to no_dwelling_12m = 1: PASS"
)


# ============================================================
# 9. Temporal provenance QC, if available
# ============================================================

print("\n9. TEMPORAL PROVENANCE QC")
print("-" * 100)

if "latest_available_consent_period" in provenance_cols:

    periods_per_transition = (
        construction_features_final
        .groupby("transition")[
            "latest_available_consent_period"
        ]
        .nunique()
    )

    print(
        "Maximum distinct consent periods "
        "within a transition:",
        periods_per_transition.max()
    )

    assert (
        periods_per_transition == 1
    ).all()

    print(
        "One latest available consent period "
        "per transition: PASS"
    )

else:
    print(
        "latest_available_consent_period is not stored "
        "in the current feature table."
    )
    print(
        "Temporal alignment remains documented upstream; "
        "no new temporal field is reconstructed here."
    )


if (
    "anchor_date" in provenance_cols
    and
    "release_date" in provenance_cols
):

    anchor = pd.to_datetime(
        construction_features_final[
            "anchor_date"
        ]
    )

    release = pd.to_datetime(
        construction_features_final[
            "release_date"
        ]
    )

    leakage_count = int(
        (release > anchor).sum()
    )

    print(
        "Release dates after anchor date:",
        leakage_count
    )

    assert leakage_count == 0

    print(
        "Point-in-time release-date leakage check: PASS"
    )


# ============================================================
# 10. Sort deterministically before persistence
# ============================================================

construction_features_final[
    "_transition_order"
] = (
    construction_features_final[
        "transition"
    ]
    .str.extract(
        r"(\d+)",
        expand=False
    )
    .astype(int)
)

construction_features_final = (
    construction_features_final
    .sort_values(
        [
            "_transition_order",
            "ta_code"
        ]
    )
    .drop(
        columns="_transition_order"
    )
    .reset_index(drop=True)
)

print("\n10. DETERMINISTIC SORT")
print("-" * 100)

print(
    "Sorted by transition number and TA code: PASS"
)


# ============================================================
# 11. Determine output directory
#
# Uses PROJECT_ROOT if already defined by config.py.
# ============================================================

if "PROJECT_ROOT" in globals():

    project_root = Path(PROJECT_ROOT)

elif "ROOT" in globals():

    project_root = Path(ROOT)

else:

    raise NameError(
        "PROJECT_ROOT or ROOT is not available. "
        "Run your project config cell first."
    )


output_dir = (
    project_root
    / "02_intermediate"
    / "construction"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir
    / "construction_features_ta_transition_frozen.parquet"
)

print("\n11. OUTPUT PATH")
print("-" * 100)

print(output_path)


# ============================================================
# 12. Save
# ============================================================

construction_features_final.to_parquet(
    output_path,
    index=False
)

assert output_path.exists()

print("\nParquet saved: PASS")


# ============================================================
# 13. Reload from disk
# ============================================================

construction_features_reloaded = (
    pd.read_parquet(output_path)
)

print("\n12. SAVE / RELOAD QC")
print("-" * 100)

print(
    "Saved rows:",
    len(construction_features_final)
)

print(
    "Reloaded rows:",
    len(construction_features_reloaded)
)

print(
    "Saved columns:",
    construction_features_final.shape[1]
)

print(
    "Reloaded columns:",
    construction_features_reloaded.shape[1]
)

assert (
    construction_features_final.shape
    ==
    construction_features_reloaded.shape
)

assert (
    list(construction_features_final.columns)
    ==
    list(construction_features_reloaded.columns)
)

print("Shape and column order preserved: PASS")


# ============================================================
# 14. Exact save/reload equality
# ============================================================

pd.testing.assert_frame_equal(
    construction_features_final,
    construction_features_reloaded,
    check_dtype=True,
    check_exact=True
)

print(
    "Exact dataframe equality after reload: PASS"
)


# ============================================================
# 15. Final frozen table preview
# ============================================================

print("\n13. FINAL TABLE PREVIEW")
print("-" * 100)

display(
    construction_features_reloaded.head(10)
)


# ============================================================
# 16. Final status
# ============================================================

print("\n" + "=" * 100)
print("FINAL TA × TRANSITION CONSTRUCTION FEATURE TABLE: PASS")
print("=" * 100)

print(
    "Rows:",
    len(construction_features_reloaded)
)

print(
    "Transitions:",
    construction_features_reloaded[
        "transition"
    ].nunique()
)

print(
    "Eligible TAs:",
    construction_features_reloaded[
        "ta_code"
    ].nunique()
)

print(
    "Primary construction features:",
    len(primary_features)
)

print(
    "Robustness features:",
    len(robustness_features)
)

print(
    "Technical indicators:",
    len(technical_cols)
)

print(
    "Unexpected missing primary values:",
    0
)

print(
    "Expected undefined composition values:",
    construction_features_reloaded[
        "construction_multiunit_share_12m"
    ].isna().sum()
)

print(
    "\nSaved to:"
)
print(output_path)

print(
    "\nSTEP 21.8 STATUS: CONSTRUCTION FEATURE TABLE FROZEN"
)